In [25]:
import os
import numpy as np

input_path = "/database/iyj0121/dataset/4k_vnt/ckpt/finetune/samsung_s4_ev2/SN_20250903_231954_03_4000x3000.raw"
output_path ="/database/iyj0121/dataset/4k_vnt/ckpt/finetune/samsung_s4_ev2/SN_20250903_231954_03_4000x3000_rggb.raw"

width = 4000
height = 3000
dtype = np.uint16

def load_raw(path, height, width, dtype=np.uint16):
    raw = np.fromfile(path, dtype=dtype)
    expected = height * width
    if raw.size != expected:
        raise ValueError(f"RAW size mismatch: got {raw.size}, expected {expected} ({height}x{width})")
    return raw.reshape(height, width)

def save_raw(path, raw):
    raw.astype(np.uint16).tofile(path)

def gbrg_to_rggb_by_row_shift(raw_gbrg):
    """
    GBRG:
        G B
        R G

    RGGB:
        R G
        G B

    GBRG -> RGGB 는 1 row upward shift로 맞출 수 있음.
    마지막 row는 직전 row 복제.
    """
    h, w = raw_gbrg.shape
    raw_rggb = np.empty_like(raw_gbrg)

    raw_rggb[:-1, :] = raw_gbrg[1:, :]
    raw_rggb[-1, :] = raw_gbrg[-1, :]   # 마지막 줄 복제

    return raw_rggb

raw_gbrg = load_raw(input_path, height, width, dtype)
raw_rggb = gbrg_to_rggb_by_row_shift(raw_gbrg)
save_raw(output_path, raw_rggb)

print(f"Saved RGGB-style raw to: {output_path}")
print("Input shape :", raw_gbrg.shape)
print("Output shape:", raw_rggb.shape)
print("dtype       :", raw_rggb.dtype)

Saved RGGB-style raw to: /database/iyj0121/dataset/4k_vnt/ckpt/finetune/samsung_s4_ev2/SN_20250903_231954_03_4000x3000_rggb.raw
Input shape : (3000, 4000)
Output shape: (3000, 4000)
dtype       : uint16


In [29]:
import os
import numpy as np

input_path = "/database/iyj0121/dataset/4k_vnt/ckpt/finetune/samsung_denoise_frame_rggb_s4_ev6_e200/denoised_raw_frame_000001.raw"
output_path ="/database/iyj0121/dataset/4k_vnt/ckpt/finetune/samsung_denoise_frame_rggb_s4_ev6_e200/denoised_raw_frame_000001_gbrg.raw"

width = 4000
height = 3000
dtype = np.uint16

def load_raw(path, height, width, dtype=np.uint16):
    raw = np.fromfile(path, dtype=dtype)
    expected = height * width
    if raw.size != expected:
        raise ValueError(f"RAW size mismatch: got {raw.size}, expected {expected} ({height}x{width})")
    return raw.reshape(height, width)

def save_raw(path, raw):
    raw.astype(np.uint16).tofile(path)

def rggb_to_gbrg_by_row_shift(raw_rggb):
    """
    gbrg_to_rggb_by_row_shift의 역방향.
    1 row downward shift.
    첫 줄은 둘째 줄 복제.
    """
    h, w = raw_rggb.shape
    raw_gbrg = np.empty_like(raw_rggb)

    raw_gbrg[1:, :] = raw_rggb[:-1, :]
    raw_gbrg[0, :] = raw_rggb[0, :]   # 첫 줄 복제

    return raw_gbrg

raw_rggb = load_raw(input_path, height, width, dtype)
raw_gbrg = rggb_to_gbrg_by_row_shift(raw_rggb)
raw_gbrg = np.clip(raw_gbrg, 0, 4095)
save_raw(output_path, raw_gbrg)

print(f"Saved restored GBRG raw to: {output_path}")
print("Input shape :", raw_rggb.shape)
print("Output shape:", raw_gbrg.shape)
print("dtype       :", raw_gbrg.dtype)

Saved restored GBRG raw to: /database/iyj0121/dataset/4k_vnt/ckpt/finetune/samsung_denoise_frame_rggb_s4_ev6_e200/denoised_raw_frame_000001_gbrg.raw
Input shape : (3000, 4000)
Output shape: (3000, 4000)
dtype       : uint16


In [30]:
import numpy as np
import cv2

input_path = "/database/iyj0121/dataset/4k_vnt/ckpt/finetune/samsung_s4_project/20250903_231958_OUTPUT_4000x3000.nv21"
output_path = "/database/iyj0121/dataset/4k_vnt/ckpt/finetune/samsung_s4_project/20250903_231958_OUTPUT_4000x3000_nv21.png"

width = 4000
height = 3000

# NV21 크기 = H*W*1.5
frame_size = int(width * height * 3 / 2)

with open(input_path, "rb") as f:
    nv21 = np.fromfile(f, dtype=np.uint8)

if nv21.size != frame_size:
    raise ValueError(f"Size mismatch: got {nv21.size}, expected {frame_size}")

# NV21 → (H*1.5, W) 형태로 reshape
nv21 = nv21.reshape((height * 3 // 2, width))

# NV21 → BGR
bgr = cv2.cvtColor(nv21, cv2.COLOR_YUV2BGR_NV21)

# 저장
cv2.imwrite(output_path, bgr)

print("Saved:", output_path)

Saved: /database/iyj0121/dataset/4k_vnt/ckpt/finetune/samsung_s4_project/20250903_231958_OUTPUT_4000x3000_nv21.png


In [31]:
import numpy as np
import cv2

input_path = "/database/iyj0121/dataset/4k_vnt/ckpt/finetune/samsung_s4_project/object_detection_1000x750.yuv"
output_path = "/database/iyj0121/dataset/4k_vnt/ckpt/finetune/samsung_s4_project/object_detection_1000x750.png"

width = 1000
height = 750

# YUV420p frame size
frame_size = width * height * 3 // 2

# 파일 읽기
with open(input_path, "rb") as f:
    yuv = np.fromfile(f, dtype=np.uint8)

if yuv.size != frame_size:
    raise ValueError(f"Size mismatch: got {yuv.size}, expected {frame_size}")

# YUV420p → (H*1.5, W)
yuv = yuv.reshape((height * 3 // 2, width))

# YUV420p → BGR
bgr = cv2.cvtColor(yuv, cv2.COLOR_YUV2BGR_I420)

# 저장
cv2.imwrite(output_path, bgr)

print("Saved:", output_path)

Saved: /database/iyj0121/dataset/4k_vnt/ckpt/finetune/samsung_s4_project/object_detection_1000x750.png


In [32]:
import numpy as np
import cv2
import os

input_path = "/database/iyj0121/dataset/4k_vnt/ckpt/finetune/samsung_s4_project/object_detection_1000x750.yuv"
output_path = "/database/iyj0121/dataset/4k_vnt/ckpt/finetune/samsung_s4_project/object_detection_1000x750.mp4"

width = 1000
height = 750
fps = 30

frame_size = width * height * 3 // 2

# 전체 파일 읽기
with open(input_path, "rb") as f:
    raw = np.fromfile(f, dtype=np.uint8)

num_frames = raw.size // frame_size
print("Total frames:", num_frames)

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

for i in range(num_frames):

    frame = raw[i*frame_size:(i+1)*frame_size]

    yuv = frame.reshape((height * 3 // 2, width))

    bgr = cv2.cvtColor(yuv, cv2.COLOR_YUV2BGR_I420)

    writer.write(bgr)

writer.release()

print("Saved:", output_path)

Total frames: 1
Saved: /database/iyj0121/dataset/4k_vnt/ckpt/finetune/samsung_s4_project/object_detection_1000x750.mp4


GT 생성 코드

In [67]:
import os
import re
import numpy as np

# =========================
# 경로 설정
# =========================
root_dir = "/database2/iyj0121/samsung/sample/scene_000027/"
save_dir = "/database2/iyj0121/samsung/gt_mean/gt_mean_scene_000027/"
os.makedirs(save_dir, exist_ok=True)

# =========================
# RAW 설정
# =========================
H = 3000
W = 4000

valid_indices = {0, 4, 5, 6, 7, 8, 9, 10}
pattern = re.compile(r"^SN_\d{8}_\d{6}_(\d{2})_4000x3000\.raw$")


def load_raw12_packed(path, h, w):
    """
    MIPI RAW12 packed 형식 로드
    2 pixels -> 3 bytes

    byte0 = p0[11:4]
    byte1 = p1[11:4]
    byte2 = p1[3:0] << 4 | p0[3:0]

    복원:
    p0 = (byte0 << 4) | (byte2 & 0x0F)
    p1 = (byte1 << 4) | (byte2 >> 4)
    """
    expected_bytes = h * w * 3 // 2
    file_size = os.path.getsize(path)

    if file_size != expected_bytes:
        raise ValueError(
            f"Size mismatch: {path}\n"
            f"  file_size(bytes) = {file_size}\n"
            f"  expected(bytes)  = {expected_bytes}"
        )

    data = np.fromfile(path, dtype=np.uint8)

    if data.size % 3 != 0:
        raise ValueError(f"Packed RAW12 data size is not divisible by 3: {path}")

    data = data.reshape(-1, 3)

    b0 = data[:, 0].astype(np.uint16)
    b1 = data[:, 1].astype(np.uint16)
    b2 = data[:, 2].astype(np.uint16)

    p0 = (b0 << 4) | (b2 & 0x0F)
    p1 = (b1 << 4) | (b2 >> 4)

    raw = np.empty(p0.size + p1.size, dtype=np.uint16)
    raw[0::2] = p0
    raw[1::2] = p1

    if raw.size != h * w:
        raise ValueError(
            f"Unpacked pixel count mismatch: {path}\n"
            f"  unpacked pixels = {raw.size}\n"
            f"  expected pixels = {h*w}"
        )

    return raw.reshape(h, w)


# =========================
# 파일 수집
# =========================
all_raw_paths = []
sub_dirs = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])

for sub in sub_dirs:
    sub_path = os.path.join(root_dir, sub)
    selected = []

    for name in sorted(os.listdir(sub_path)):
        if not name.endswith(".raw"):
            continue

        m = pattern.match(name)
        if m is None:
            print(f"[Skip - pattern mismatch] {os.path.join(sub_path, name)}")
            continue

        frame_idx = int(m.group(1))
        if frame_idx in valid_indices:
            selected.append(os.path.join(sub_path, name))

    if len(selected) != 8:
        print(f"[Warning] {sub} has {len(selected)} selected images (expected 8)")
        for p in selected:
            print(f"   - {os.path.basename(p)}")

    all_raw_paths.extend(selected)

print(f"Total selected images: {len(all_raw_paths)}")

if len(all_raw_paths) != 80:
    raise RuntimeError(f"Expected 80 images, but got {len(all_raw_paths)}")


# =========================
# Mean GT 생성
# =========================
acc = np.zeros((H, W), dtype=np.float64)

for i, path in enumerate(all_raw_paths, 1):
    img = load_raw12_packed(path, H, W).astype(np.float64)
    acc += img

    if i % 10 == 0 or i == len(all_raw_paths):
        print(f"{i}/{len(all_raw_paths)} processed")

mean_img = acc / len(all_raw_paths)

# 12bit 범위로 클리핑
mean_img = np.clip(mean_img, 0, 4095)

# GT를 uint16로 저장
mean_img_u16 = mean_img.astype(np.uint16)

save_path = os.path.join(save_dir, "scene_000027_gt_mean_80frames_u16.raw")
mean_img_u16.tofile(save_path)

print(f"GT saved to: {save_path}")
print(f"GT dtype: {mean_img_u16.dtype}, shape: {mean_img_u16.shape}, min: {mean_img_u16.min()}, max: {mean_img_u16.max()}")

Total selected images: 80
10/80 processed
20/80 processed
30/80 processed
40/80 processed
50/80 processed
60/80 processed
70/80 processed
80/80 processed
GT saved to: /database2/iyj0121/samsung/gt_mean/gt_mean_scene_000027/scene_000027_gt_mean_80frames_u16.raw
GT dtype: uint16, shape: (3000, 4000), min: 223, max: 1668


극저조도용 GT 생성 코드

In [51]:
import os
import re
import numpy as np

# =========================
# 경로 설정
# =========================
root_dir = "/database2/iyj0121/samsung/sample/scene_000012"
save_dir = "/database2/iyj0121/samsung/gt_mean/gt_mean_scene_000012/"
os.makedirs(save_dir, exist_ok=True)

# =========================
# RAW 설정
# =========================
H = 3000
W = 4000

valid_indices = {0, 4, 5, 6, 7, 8}
pattern = re.compile(r"^SN_\d{8}_\d{6}_(\d{2})_4000x3000\.raw$")


def load_raw12_packed(path, h, w):
    """
    MIPI RAW12 packed 형식 로드
    2 pixels -> 3 bytes

    byte0 = p0[11:4]
    byte1 = p1[11:4]
    byte2 = p1[3:0] << 4 | p0[3:0]

    복원:
    p0 = (byte0 << 4) | (byte2 & 0x0F)
    p1 = (byte1 << 4) | (byte2 >> 4)
    """
    expected_bytes = h * w * 3 // 2
    file_size = os.path.getsize(path)

    if file_size != expected_bytes:
        raise ValueError(
            f"Size mismatch: {path}\n"
            f"  file_size(bytes) = {file_size}\n"
            f"  expected(bytes)  = {expected_bytes}"
        )

    data = np.fromfile(path, dtype=np.uint8)

    if data.size % 3 != 0:
        raise ValueError(f"Packed RAW12 data size is not divisible by 3: {path}")

    data = data.reshape(-1, 3)

    b0 = data[:, 0].astype(np.uint16)
    b1 = data[:, 1].astype(np.uint16)
    b2 = data[:, 2].astype(np.uint16)

    p0 = (b0 << 4) | (b2 & 0x0F)
    p1 = (b1 << 4) | (b2 >> 4)

    raw = np.empty(p0.size + p1.size, dtype=np.uint16)
    raw[0::2] = p0
    raw[1::2] = p1

    if raw.size != h * w:
        raise ValueError(
            f"Unpacked pixel count mismatch: {path}\n"
            f"  unpacked pixels = {raw.size}\n"
            f"  expected pixels = {h*w}"
        )

    return raw.reshape(h, w)


# =========================
# 파일 수집
# =========================
all_raw_paths = []
sub_dirs = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])

for sub in sub_dirs:
    sub_path = os.path.join(root_dir, sub)
    selected = []

    for name in sorted(os.listdir(sub_path)):
        if not name.endswith(".raw"):
            continue

        m = pattern.match(name)
        if m is None:
            print(f"[Skip - pattern mismatch] {os.path.join(sub_path, name)}")
            continue

        frame_idx = int(m.group(1))
        if frame_idx in valid_indices:
            selected.append(os.path.join(sub_path, name))

    if len(selected) != 6:
        print(f"[Warning] {sub} has {len(selected)} selected images (expected 8)")
        for p in selected:
            print(f"   - {os.path.basename(p)}")

    all_raw_paths.extend(selected)

print(f"Total selected images: {len(all_raw_paths)}")

if len(all_raw_paths) != 60:
    raise RuntimeError(f"Expected 80 images, but got {len(all_raw_paths)}")


# =========================
# Mean GT 생성
# =========================
acc = np.zeros((H, W), dtype=np.float64)

for i, path in enumerate(all_raw_paths, 1):
    img = load_raw12_packed(path, H, W).astype(np.float64)
    acc += img

    if i % 10 == 0 or i == len(all_raw_paths):
        print(f"{i}/{len(all_raw_paths)} processed")

mean_img = acc / len(all_raw_paths)

# 12bit 범위로 클리핑
mean_img = np.clip(mean_img, 0, 4095)

# GT를 uint16로 저장
mean_img_u16 = mean_img.astype(np.uint16)

save_path = os.path.join(save_dir, "scene_000012_gt_mean_80frames_u16.raw")
mean_img_u16.tofile(save_path)

print(f"GT saved to: {save_path}")
print(f"GT dtype: {mean_img_u16.dtype}, shape: {mean_img_u16.shape}, min: {mean_img_u16.min()}, max: {mean_img_u16.max()}")

Total selected images: 60
10/60 processed
20/60 processed
30/60 processed
40/60 processed
50/60 processed
60/60 processed
GT saved to: /database2/iyj0121/samsung/gt_mean/gt_mean_scene_000012/scene_000012_gt_mean_80frames_u16.raw
GT dtype: uint16, shape: (3000, 4000), min: 222, max: 3651


EV별 GT 생성 코드

In [2]:
import os
import re
import glob
import numpy as np

# =========================
# 경로 설정
# =========================
sample_root = "/database2/iyj0121/samsung/251205/"
save_root = "/database2/iyj0121/samsung/251205_gt_mean"
os.makedirs(save_root, exist_ok=True)

# =========================
# 데이터 설정
# =========================
HEIGHT = 3000
WIDTH = 4000

SCENE_START = 13
SCENE_END = 13

# index -> ev 이름 매핑
EV_INDEX_MAP = {
    1: "evm6",   # ev -6
    2: "evm4",   # ev -4
    3: "evm2",   # ev -2
}

# 파일명 예:
# SN_20251120_101116_01_4000x3000.raw
FILE_PATTERN = re.compile(r"^SN_\d{8}_\d{6}_(\d{2})_4000x3000\.raw$")


def load_raw12_packed(path, h, w):
    """
    MIPI RAW12 packed
    2 pixels -> 3 bytes

    unpack:
    p0 = (b0 << 4) | (b2 & 0x0F)
    p1 = (b1 << 4) | (b2 >> 4)
    """
    expected_bytes = h * w * 3 // 2
    file_size = os.path.getsize(path)

    if file_size != expected_bytes:
        raise ValueError(
            f"RAW12 packed size mismatch: {path}\n"
            f"  file_size(bytes) = {file_size}\n"
            f"  expected(bytes)  = {expected_bytes}"
        )

    data = np.fromfile(path, dtype=np.uint8)

    if data.size % 3 != 0:
        raise ValueError(f"Packed RAW12 data size is not divisible by 3: {path}")

    data = data.reshape(-1, 3)

    b0 = data[:, 0].astype(np.uint16)
    b1 = data[:, 1].astype(np.uint16)
    b2 = data[:, 2].astype(np.uint16)

    p0 = (b0 << 4) | (b2 & 0x0F)
    p1 = (b1 << 4) | (b2 >> 4)

    raw = np.empty(p0.size + p1.size, dtype=np.uint16)
    raw[0::2] = p0
    raw[1::2] = p1

    expected_pixels = h * w
    if raw.size != expected_pixels:
        raise ValueError(
            f"Unpacked pixel count mismatch: {path}\n"
            f"  unpacked pixels = {raw.size}\n"
            f"  expected pixels = {expected_pixels}"
        )

    return raw.reshape(h, w)


def collect_ev_files_for_scene(scene_dir, target_idx):
    """
    scene_dir 아래의 timestamp 하위 폴더들을 돌면서
    target_idx(1,2,3)에 해당하는 raw 파일만 수집
    """
    selected = []

    subdirs = sorted([
        os.path.join(scene_dir, d)
        for d in os.listdir(scene_dir)
        if os.path.isdir(os.path.join(scene_dir, d))
    ])

    for subdir in subdirs:
        for fname in sorted(os.listdir(subdir)):
            if not fname.endswith(".raw"):
                continue

            m = FILE_PATTERN.match(fname)
            if m is None:
                continue

            frame_idx = int(m.group(1))
            if frame_idx == target_idx:
                selected.append(os.path.join(subdir, fname))

    return selected


def mean_raw_list(raw_paths, h, w):
    if len(raw_paths) == 0:
        raise RuntimeError("No raw files to average.")

    acc = np.zeros((h, w), dtype=np.float64)

    for i, path in enumerate(raw_paths, 1):
        raw = load_raw12_packed(path, h, w).astype(np.float64)
        acc += raw

    mean_img = acc / len(raw_paths)
    mean_img = np.clip(mean_img, 0, 4095).astype(np.uint16)
    return mean_img


def save_mean_gt(scene_id, ev_name, mean_img, save_root):
    scene_name = f"scene_{scene_id:06d}"
    save_dir = os.path.join(save_root, f"ev_gt_mean_{scene_name}")
    os.makedirs(save_dir, exist_ok=True)

    save_name = f"{scene_name}_{ev_name}_gt_mean_{mean_img.shape[0] and '30frames'}_u16.raw"
    save_path = os.path.join(save_dir, save_name)

    mean_img.tofile(save_path)
    return save_path


def main():
    for scene_id in range(SCENE_START, SCENE_END + 1):
        scene_name = f"scene_{scene_id:06d}"
        scene_dir = os.path.join(sample_root, scene_name)

        if not os.path.isdir(scene_dir):
            print(f"[Skip] Scene folder not found: {scene_dir}")
            continue

        print(f"\n===== Processing {scene_name} =====")

        for ev_idx, ev_name in EV_INDEX_MAP.items():
            ev_files = collect_ev_files_for_scene(scene_dir, ev_idx)

            print(f"[{scene_name}] idx={ev_idx:02d} ({ev_name}) -> {len(ev_files)} files")

            if len(ev_files) == 0:
                print(f"  [Skip] No files found for idx={ev_idx:02d}") 
                continue

            if len(ev_files) != 30:
                print(f"  [Warning] Expected 30 files, but found {len(ev_files)}")

            mean_img = mean_raw_list(ev_files, HEIGHT, WIDTH)
            save_path = save_mean_gt(scene_id, ev_name, mean_img, save_root)

            print(f"  Saved: {save_path}")
            print(f"  dtype={mean_img.dtype}, min={mean_img.min()}, max={mean_img.max()}")


if __name__ == "__main__":
    main()


===== Processing scene_000013 =====
[scene_000013] idx=01 (evm6) -> 30 files
  Saved: /database2/iyj0121/samsung/251205_gt_mean/ev_gt_mean_scene_000013/scene_000013_evm6_gt_mean_30frames_u16.raw
  dtype=uint16, min=244, max=391
[scene_000013] idx=02 (evm4) -> 30 files
  Saved: /database2/iyj0121/samsung/251205_gt_mean/ev_gt_mean_scene_000013/scene_000013_evm4_gt_mean_30frames_u16.raw
  dtype=uint16, min=245, max=463
[scene_000013] idx=03 (evm2) -> 30 files
  Saved: /database2/iyj0121/samsung/251205_gt_mean/ev_gt_mean_scene_000013/scene_000013_evm2_gt_mean_30frames_u16.raw
  dtype=uint16, min=225, max=984


16bit .raw 시각화 코드

In [ ]:
import os
import numpy as np
import torch
import cv2
import rawpy

from debayer import Debayer5x5, Layout 

def postprocess(rgb):
    rgb = rgb.cpu().detach().numpy().astype(np.float32)
    rgb = np.transpose(rgb, (0,2,3,1))
    rgb = np.clip(rgb, 0, 1)
    rgb = np.uint8(rgb*255)
    return rgb

debayer = Debayer5x5(layout=Layout.GBRG).cuda(device=2)

def raw_to_png(raw_file_path, save_folder):
    savename = os.path.basename(raw_file_path).replace('.raw', '.png')
    
    with open(raw_file_path, 'rb') as f:
        raw = np.fromfile(f, dtype=np.uint16)

    
    #raw = raw.reshape(512, 512)  
    #raw = raw.reshape(2160, 3840)
    raw = raw.reshape(3000, 4000)
    #raw = raw.reshape(896, 448)
    #raw = rawpy.imread(raw_file_path).raw_image_visible
    #raw = raw.reshape(2848, 4256)
    
    raw = raw.astype(np.float32)
    raw = (raw - 255) / ((2**12 - 1) - 255)
    #raw = (raw-512) / ((2**16 - 1)-512)
    #raw = raw / 65535.0
    raw = raw * 20.0 #gain
    raw = torch.from_numpy(raw).unsqueeze(0).unsqueeze(0).cuda(device=2)
    #raw = SpaceToDepth_fact2(raw)

    with torch.no_grad():
        aa = torch.clamp(raw, min=0.0, max=1.0)
        aa = torch.clamp(aa, min=1e-8) ** (1.0 / 2.2)

        aa = postprocess(debayer(aa.clone().detach()))[0]
        
        cv2.imwrite(os.path.join(save_folder, savename), cv2.cvtColor(aa, cv2.COLOR_BGR2RGB))

    print(f'Saved PNG and RAW files for {savename}') #nafnet_120000iter

raw_file_path ="/database2/iyj0121/samsung/sogang/resultrestormer_s4_iter120000_train_ev0/SN_20250903_231954_03_4000x3000.raw.raw"
save_folder = "/database2/iyj0121/samsung/sogang/resultrestormer_s4_iter120000_train_ev0/"

if not os.path.exists(save_folder):
    os.makedirs(save_folder)

raw_to_png(raw_file_path, save_folder)


Saved PNG and RAW files for SN_20250903_231954_03_4000x3000.png.png


In [38]:
import os
import numpy as np
import tifffile
import cv2

# =========================
# 경로 설정
# =========================
dng_path = "/database2/iyj0121/samsung/sogang/bias_raw_cap/2356/50ms/debug/20260403_133617_ISO2356_1000us_001.dng"
png_path = dng_path.replace(".dng", "_color_gain100.png")

# =========================
# DNG RAW 읽기
# =========================
with tifffile.TiffFile(dng_path) as tif:
    raw = tif.pages[0].asarray().astype(np.float32)

print("RAW shape :", raw.shape)
print("RAW min   :", raw.min())
print("RAW max   :", raw.max())
print("RAW mean  :", raw.mean())
print("RAW std   :", raw.std())

# =========================
# 정규화 파라미터
# =========================
BLACK_LEVEL = 64.0
WHITE_LEVEL = 1023.0
LINEAR_GAIN = 1.0

# =========================
# black level 제거 후 0~1 정규화
# =========================
raw_norm = (raw - BLACK_LEVEL) / (WHITE_LEVEL - BLACK_LEVEL)
raw_norm = np.clip(raw_norm, 0.0, 1.0)

# =========================
# linear gain 적용
# =========================
raw_gain = raw_norm * LINEAR_GAIN
raw_gain = np.clip(raw_gain, 0.0, 1.0)

# =========================
# 16bit Bayer 이미지로 변환
# OpenCV demosaic 품질을 위해 16bit 유지
# =========================
bayer_u16 = (raw_gain * 65535.0).astype(np.uint16)

# =========================
# Bayer -> Color demosaic
# 메타데이터 기준 CFAPattern = [G,B][R,G] -> GBRG
# OpenCV 코드도 GB 사용
# =========================
color_u16 = cv2.cvtColor(bayer_u16, cv2.COLOR_BAYER_GB2BGR)

# =========================
# PNG 저장용 8bit 변환
# =========================
color_u8 = np.clip(color_u16 / 256.0, 0, 255).astype(np.uint8)

# =========================
# 저장
# =========================
ok = cv2.imwrite(png_path, color_u8)
if not ok:
    raise RuntimeError(f"Failed to save PNG: {png_path}")

print("\nSaved PNG:", png_path)
print("Color min :", color_u8.min())
print("Color max :", color_u8.max())
print("Color mean:", color_u8.mean())
print("Color std :", color_u8.std())

TiffTag 274: 9 is not a valid ORIENTATION


RAW shape : (3060, 4080)
RAW min   : 42.0
RAW max   : 1023.0
RAW mean  : 212.93533
RAW std   : 202.28703

Saved PNG: /database2/iyj0121/samsung/sogang/bias_raw_cap/2356/50ms/debug/20260403_133617_ISO2356_1000us_001_color_gain100.png
Color min : 0
Color max : 255
Color mean: 35.27507395659789
Color std : 48.805386248224494


sample 폴더 MIPI RAW12 시각화 코드

In [66]:
import os
import numpy as np
import torch
import cv2

from debayer import Debayer5x5, Layout


def postprocess(rgb):
    rgb = rgb.detach().cpu().numpy().astype(np.float32)
    rgb = np.transpose(rgb, (0, 2, 3, 1))
    rgb = np.clip(rgb, 0, 1)
    rgb = np.uint8(rgb * 255.0)
    return rgb


def load_raw12_packed(path, height, width):
    """
    MIPI RAW12 packed
    2 pixels -> 3 bytes

    unpack:
    p0 = (b0 << 4) | (b2 & 0x0F)
    p1 = (b1 << 4) | (b2 >> 4)
    """
    expected_bytes = height * width * 3 // 2
    file_size = os.path.getsize(path)

    if file_size != expected_bytes:
        raise ValueError(
            f"RAW12 packed size mismatch: {path}\n"
            f"  file_size(bytes) = {file_size}\n"
            f"  expected(bytes)  = {expected_bytes}"
        )

    data = np.fromfile(path, dtype=np.uint8)

    if data.size % 3 != 0:
        raise ValueError(f"Packed RAW12 size is not divisible by 3: {path}")

    data = data.reshape(-1, 3)

    b0 = data[:, 0].astype(np.uint16)
    b1 = data[:, 1].astype(np.uint16)
    b2 = data[:, 2].astype(np.uint16)

    p0 = (b0 << 4) | (b2 & 0x0F)
    p1 = (b1 << 4) | (b2 >> 4)

    raw = np.empty(p0.size + p1.size, dtype=np.uint16)
    raw[0::2] = p0
    raw[1::2] = p1

    expected_pixels = height * width
    if raw.size != expected_pixels:
        raise ValueError(
            f"Unpacked pixel count mismatch: {path}\n"
            f"  unpacked pixels = {raw.size}\n"
            f"  expected pixels = {expected_pixels}"
        )

    return raw.reshape(height, width)


def load_raw_unpacked_u16(path, height, width):
    raw = np.fromfile(path, dtype=np.uint16)
    expected = height * width
    if raw.size != expected:
        raise ValueError(
            f"Unpacked uint16 RAW size mismatch: {path}\n"
            f"  raw.size = {raw.size}\n"
            f"  expected = {expected}"
        )
    return raw.reshape(height, width)


def load_raw_auto(path, height, width):
    """
    파일 크기로 자동 판별
    - packed RAW12: 18,000,000 bytes
    - unpacked uint16: 24,000,000 bytes
    """
    file_size = os.path.getsize(path)
    packed_bytes = height * width * 3 // 2
    unpacked_bytes = height * width * 2

    if file_size == packed_bytes:
        print(f"[Info] Detected packed RAW12: {path}")
        return load_raw12_packed(path, height, width)
    elif file_size == unpacked_bytes:
        print(f"[Info] Detected unpacked uint16 RAW: {path}")
        return load_raw_unpacked_u16(path, height, width)
    else:
        raise ValueError(
            f"Unknown RAW format: {path}\n"
            f"  file_size(bytes) = {file_size}\n"
            f"  packed_bytes     = {packed_bytes}\n"
            f"  unpacked_bytes   = {unpacked_bytes}"
        )


def raw_to_png(
    raw_file_path,
    save_folder,
    height=3000,
    width=4000,
    black_level=256,
    white_level=4095,
    gain=100.0,
    gamma=2.2,
    layout=Layout.GBRG,
    device=0,
    mode="auto",  # "auto", "packed", "unpacked"
):
    os.makedirs(save_folder, exist_ok=True)

    savename = os.path.basename(raw_file_path).replace(".raw", ".png")

    # 1. RAW 로드
    if mode == "packed":
        raw = load_raw12_packed(raw_file_path, height, width)
    elif mode == "unpacked":
        raw = load_raw_unpacked_u16(raw_file_path, height, width)
    elif mode == "auto":
        raw = load_raw_auto(raw_file_path, height, width)
    else:
        raise ValueError(f"Unknown mode: {mode}")

    # 2. normalize
    raw = raw.astype(np.float32)
    raw = (raw - black_level) / float(white_level - black_level)
    raw = np.clip(raw, 0.0, 1.0)

    # 3. gain
    raw = raw * gain
    raw = np.clip(raw, 0.0, 1.0)

    # 4. torch tensor
    raw = torch.from_numpy(raw).unsqueeze(0).unsqueeze(0).cuda(device=device)

    debayer = Debayer5x5(layout=layout).cuda(device=device)

    # 5. gamma + debayer
    with torch.no_grad():
        x = torch.clamp(raw, min=1e-8, max=1.0)
        x = x ** (1.0 / gamma)

        rgb = debayer(x)
        rgb = postprocess(rgb)[0]

        save_path = os.path.join(save_folder, savename)
        cv2.imwrite(save_path, cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))

    print(f"[Saved] {save_path}")


if __name__ == "__main__":
    # sample 폴더의 packed RAW12 예시
    raw_file_path = "/database2/iyj0121/samsung/251205/scene_000019/SN_20251205_134517/SN_20251205_134517_01_4000x3000.raw"
    save_folder = "/database2/iyj0121/samsung/251205/scene_000019/SN_20251205_134517/"

    raw_to_png(
        raw_file_path=raw_file_path,
        save_folder=save_folder,
        height=3000,
        width=4000,
        black_level=256,
        white_level=4095,
        gain=50.0,
        gamma=2.2,
        layout=Layout.GBRG,
        device=1,
        mode="auto"   # sample packed도 되고, unpacked raw도 같이 처리 가능
    )

[Info] Detected packed RAW12: /database2/iyj0121/samsung/251205/scene_000019/SN_20251205_134517/SN_20251205_134517_01_4000x3000.raw
[Saved] /database2/iyj0121/samsung/251205/scene_000019/SN_20251205_134517/SN_20251205_134517_01_4000x3000.png


GT 시각화 코드

In [1]:
import os
import numpy as np
import torch
import cv2

from debayer import Debayer5x5, Layout 

# =========================
# 경로 설정
# =========================
GT_ROOT = "/database2/iyj0121/samsung/251205_gt_mean"
SAVE_ROOT = "/database2/iyj0121/samsung/ev_gt_mean_scene_251205_sRGB_evm4"
os.makedirs(SAVE_ROOT, exist_ok=True)

SCENE_START = 1
SCENE_END = 19

HEIGHT = 3000
WIDTH = 4000

# =========================
# Debayer (GPU 1)
# =========================
device = torch.device("cuda:1")
debayer = Debayer5x5(layout=Layout.GBRG).to(device)


def postprocess(rgb):
    rgb = rgb.cpu().detach().numpy().astype(np.float32)
    rgb = np.transpose(rgb, (0, 2, 3, 1))
    rgb = np.clip(rgb, 0, 1)
    rgb = np.uint8(rgb * 255)
    return rgb


def raw_to_png(raw_file_path, save_path):
    # RAW 읽기
    raw = np.fromfile(raw_file_path, dtype=np.uint16)

    if raw.size != HEIGHT * WIDTH:
        raise ValueError(
            f"RAW size mismatch: {raw_file_path}\n"
            f"  raw.size = {raw.size}\n"
            f"  expected = {HEIGHT * WIDTH}"
        )

    raw = raw.reshape(HEIGHT, WIDTH)

    # normalize
    raw = raw.astype(np.float32)
    raw = (raw - 256) / (4095 - 256)
    raw = np.clip(raw, 0.0, 1.0)

    # gain (visualization용)
    raw = raw * 50.0

    # torch tensor
    raw = torch.from_numpy(raw).unsqueeze(0).unsqueeze(0).to(device)

    with torch.no_grad():
        aa = torch.clamp(raw, min=0.0, max=1.0)
        aa = torch.clamp(aa, min=1e-8) ** (1.0 / 2.2)

        # debayer
        rgb = debayer(aa.clone())

        # numpy 변환
        rgb = postprocess(rgb)[0]

        # 저장 (RGB로 변환)
        cv2.imwrite(save_path, cv2.cvtColor(rgb, cv2.COLOR_BGR2RGB))


def main():
    for scene_id in range(SCENE_START, SCENE_END + 1):
        scene_name = f"scene_{scene_id:06d}"

        raw_path = os.path.join(
            GT_ROOT,
            f"ev_gt_mean_{scene_name}",
            f"{scene_name}_evm4_gt_mean_30frames_u16.raw"
        )

        if not os.path.isfile(raw_path):
            print(f"[Skip] RAW not found: {raw_path}")
            continue

        save_name = f"{scene_name}_evm4.png"
        save_path = os.path.join(SAVE_ROOT, save_name)

        print(f"[Processing] {scene_name}")

        raw_to_png(raw_path, save_path)

        print(f"[Saved] {save_path}")


if __name__ == "__main__":
    main()

[Processing] scene_000001
[Saved] /database2/iyj0121/samsung/ev_gt_mean_scene_251205_sRGB_evm4/scene_000001_evm4.png
[Processing] scene_000002
[Saved] /database2/iyj0121/samsung/ev_gt_mean_scene_251205_sRGB_evm4/scene_000002_evm4.png
[Processing] scene_000003
[Saved] /database2/iyj0121/samsung/ev_gt_mean_scene_251205_sRGB_evm4/scene_000003_evm4.png
[Processing] scene_000004
[Saved] /database2/iyj0121/samsung/ev_gt_mean_scene_251205_sRGB_evm4/scene_000004_evm4.png
[Processing] scene_000005
[Saved] /database2/iyj0121/samsung/ev_gt_mean_scene_251205_sRGB_evm4/scene_000005_evm4.png
[Processing] scene_000006
[Saved] /database2/iyj0121/samsung/ev_gt_mean_scene_251205_sRGB_evm4/scene_000006_evm4.png
[Skip] RAW not found: /database2/iyj0121/samsung/251205_gt_mean/ev_gt_mean_scene_000007/scene_000007_evm4_gt_mean_30frames_u16.raw
[Processing] scene_000008
[Saved] /database2/iyj0121/samsung/ev_gt_mean_scene_251205_sRGB_evm4/scene_000008_evm4.png
[Processing] scene_000009
[Saved] /database2/iyj01

In [ ]:
exit()

: 

bias frame

In [32]:
import os
import json
import subprocess

DNG_PATH = "/database2/iyj0121/samsung/sogang/bias_raw_cap/2356/50ms__/20260403_135601_ISO2356_50000us_014.dng"


def print_dict(d, title=None):
    if title:
        print("\n" + "=" * 80)
        print(title)
        print("=" * 80)
    for k in sorted(d.keys()):
        print(f"{k}: {d[k]}")


def read_with_exiftool(path):
    """
    exiftool이 설치되어 있으면 가장 많은 메타데이터를 안정적으로 읽을 수 있음.
    """
    cmd = ["exiftool", "-j", path]
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)
    data = json.loads(result.stdout)
    if not data:
        raise RuntimeError("No metadata returned from exiftool.")
    return data[0]


def read_with_pillow(path):
    """
    Pillow 기반 기본 EXIF/TIFF 태그 읽기
    """
    from PIL import Image, ExifTags

    img = Image.open(path)

    meta = {}
    meta["format"] = img.format
    meta["mode"] = img.mode
    meta["size"] = img.size

    # getexif()는 일반 EXIF 태그
    try:
        exif = img.getexif()
        if exif:
            for tag_id, value in exif.items():
                tag_name = ExifTags.TAGS.get(tag_id, str(tag_id))
                meta[f"EXIF::{tag_name}"] = value
    except Exception as e:
        meta["EXIF_read_error"] = str(e)

    # TIFF tag_v2는 DNG/TIFF 계열에서 좀 더 유용할 수 있음
    try:
        if hasattr(img, "tag_v2"):
            for tag_id, value in img.tag_v2.items():
                tag_name = ExifTags.TAGS.get(tag_id, f"TIFFTag_{tag_id}")
                meta[f"TIFF::{tag_name}"] = value
    except Exception as e:
        meta["TIFF_read_error"] = str(e)

    return meta


def read_with_tifffile(path):
    """
    tifffile 기반 TIFF/DNG 태그 읽기
    """
    import tifffile

    meta = {}

    with tifffile.TiffFile(path) as tif:
        meta["num_pages"] = len(tif.pages)

        for page_idx, page in enumerate(tif.pages):
            for tag in page.tags.values():
                key = f"page{page_idx}::{tag.name}"
                try:
                    value = tag.value
                except Exception:
                    value = str(tag.valueoffset)
                meta[key] = value

    return meta


def main():
    path = DNG_PATH

    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")

    print(f"Target file: {path}")
    print(f"File size  : {os.path.getsize(path)} bytes")

    # 1) exiftool 우선
    try:
        meta = read_with_exiftool(path)
        print_dict(meta, title="DNG metadata via exiftool")
        return
    except FileNotFoundError:
        print("\n[Info] exiftool not found. Fallback to Python libraries.")
    except Exception as e:
        print(f"\n[Warning] exiftool read failed: {e}")
        print("[Info] Fallback to Python libraries.")

    # 2) Pillow 시도
    try:
        meta = read_with_pillow(path)
        print_dict(meta, title="DNG metadata via Pillow")
    except Exception as e:
        print(f"\n[Warning] Pillow read failed: {e}")

    # 3) tifffile 시도
    try:
        meta = read_with_tifffile(path)
        print_dict(meta, title="DNG metadata via tifffile")
    except Exception as e:
        print(f"\n[Warning] tifffile read failed: {e}")


if __name__ == "__main__":
    main()

Target file: /database2/iyj0121/samsung/sogang/bias_raw_cap/2356/50ms__/20260403_135601_ISO2356_50000us_014.dng
File size  : 24999544 bytes

DNG metadata via exiftool
ActiveArea: 0 0 3060 4080
Aperture: 1.7
AsShotNeutral: 0.5263671875 1 0.6201171875
BaselineExposure: 0
BitsPerSample: 16
BlackLevel: 64 64 64 64
BlackLevelRepeatDim: 2 2
CFALayout: Rectangular
CFAPattern: [Green,Blue][Red,Green]
CFAPattern2: 1 2 0 1
CFAPlaneColor: Red,Green,Blue
CFARepeatPatternDim: 2 2
CalibrationIlluminant1: D65
CalibrationIlluminant2: Standard Light A
CameraCalibration1: 0.98046875 0 0 0 1 0 0 0 1.045898438
CameraCalibration2: 0.98046875 0 0 0 1 0 0 0 1.045898438
ColorMatrix1: 0.7919921875 -0.177734375 -0.1279296875 -0.3330078125 1.298828125 0.0166015625 -0.1064453125 0.34375 0.4208984375
ColorMatrix2: 1.723632812 -1.03125 -0.2177734375 0.0205078125 1.040039062 -0.173828125 0.083984375 0.1513671875 0.423828125
Compression: Uncompressed
Copyright: 
DNGBackwardVersion: 1.1.0.0
DNGVersion: 1.4.0.0
DateTim

In [20]:
import os
import glob
import numpy as np

# =========================
# 경로 설정
# =========================
root_pattern = "/database2/iyj0121/samsung/sogang/bias_frame/SN_20260327_*"
save_dir = "/database2/iyj0121/samsung/sogang/bias_frame/mean_result"
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(save_dir, "bias_mean_90frames_u16.raw")

# =========================
# RAW 설정
# =========================
H = 3000
W = 4000


def load_raw12_packed(path, h, w):
    """
    MIPI RAW12 packed 형식 로드
    2 pixels -> 3 bytes

    저장 방식:
      byte0 = p0[11:4]
      byte1 = p1[11:4]
      byte2 = p1[3:0] << 4 | p0[3:0]

    복원:
      p0 = (byte0 << 4) | (byte2 & 0x0F)
      p1 = (byte1 << 4) | (byte2 >> 4)
    """
    expected_bytes = h * w * 3 // 2
    file_size = os.path.getsize(path)

    if file_size != expected_bytes:
        raise ValueError(
            f"Size mismatch: {path}\n"
            f"  file_size(bytes) = {file_size}\n"
            f"  expected(bytes)  = {expected_bytes}"
        )

    data = np.fromfile(path, dtype=np.uint8)

    if data.size % 3 != 0:
        raise ValueError(f"Packed RAW12 data size is not divisible by 3: {path}")

    data = data.reshape(-1, 3)

    b0 = data[:, 0].astype(np.uint16)
    b1 = data[:, 1].astype(np.uint16)
    b2 = data[:, 2].astype(np.uint16)

    p0 = (b0 << 4) | (b2 & 0x0F)
    p1 = (b1 << 4) | (b2 >> 4)

    raw = np.empty(p0.size + p1.size, dtype=np.uint16)
    raw[0::2] = p0
    raw[1::2] = p1

    if raw.size != h * w:
        raise ValueError(
            f"Unpacked pixel count mismatch: {path}\n"
            f"  unpacked pixels = {raw.size}\n"
            f"  expected pixels = {h * w}"
        )

    return raw.reshape(h, w)


# =========================
# 디렉토리 수집
# =========================
all_dirs = sorted(glob.glob(root_pattern))
all_dirs = [d for d in all_dirs if os.path.isdir(d)]

print(f"Found directories: {len(all_dirs)}")
for d in all_dirs:
    print(f"  {d}")

if len(all_dirs) != 15:
    print(f"[Warning] Expected 15 directories, but found {len(all_dirs)}")

# =========================
# 파일 수집
# =========================
all_raw_paths = []

for d in all_dirs:
    # 예: SN_20260327_123606/SN_20260327_*_0*_4000x3000.raw
    raw_pattern = os.path.join(d, "SN_20260327_*_0*_4000x3000.raw")
    raw_files = sorted(glob.glob(raw_pattern))

    if len(raw_files) != 6:
        print(f"[Warning] {d} has {len(raw_files)} files (expected 6)")
        for f in raw_files:
            print(f"   - {os.path.basename(f)}")

    all_raw_paths.extend(raw_files)

print(f"\nTotal selected RAW files: {len(all_raw_paths)}")

expected_total = 15 * 6
if len(all_raw_paths) != expected_total:
    raise RuntimeError(f"Expected {expected_total} images, but got {len(all_raw_paths)}")

# =========================
# Mean 계산
# =========================
acc = np.zeros((H, W), dtype=np.float64)

for i, path in enumerate(all_raw_paths, 1):
    img = load_raw12_packed(path, H, W).astype(np.float64)
    acc += img

    if i % 10 == 0 or i == len(all_raw_paths):
        print(f"{i}/{len(all_raw_paths)} processed")

mean_img = acc / len(all_raw_paths)

# =========================
# 12bit 범위 클리핑 후 저장
# =========================
mean_img = np.clip(mean_img, 0, 4095)
mean_img_u16 = mean_img.astype(np.uint16)

mean_img_u16.tofile(save_path)

print("\n=========================")
print(f"GT saved to: {save_path}")
print(f"GT dtype: {mean_img_u16.dtype}")
print(f"GT shape: {mean_img_u16.shape}")
print(f"GT min: {mean_img_u16.min()}")
print(f"GT max: {mean_img_u16.max()}")

# =========================
# 추가 디버깅용 통계
# =========================
first_img = load_raw12_packed(all_raw_paths[0], H, W).astype(np.float64)

print("\n[Debug statistics]")
print(f"single frame mean: {first_img.mean():.4f}")
print(f"single frame std : {first_img.std():.4f}")
print(f"mean image mean  : {mean_img.mean():.4f}")
print(f"mean image std   : {mean_img.std():.4f}")
print(f"mean image min: {mean_img.min()}")
print(f"mean image max: {mean_img.max()}")

Found directories: 15
  /database2/iyj0121/samsung/sogang/bias_frame/SN_20260327_112411
  /database2/iyj0121/samsung/sogang/bias_frame/SN_20260327_114251
  /database2/iyj0121/samsung/sogang/bias_frame/SN_20260327_114258
  /database2/iyj0121/samsung/sogang/bias_frame/SN_20260327_114303
  /database2/iyj0121/samsung/sogang/bias_frame/SN_20260327_114308
  /database2/iyj0121/samsung/sogang/bias_frame/SN_20260327_114313
  /database2/iyj0121/samsung/sogang/bias_frame/SN_20260327_123545
  /database2/iyj0121/samsung/sogang/bias_frame/SN_20260327_123552
  /database2/iyj0121/samsung/sogang/bias_frame/SN_20260327_123557
  /database2/iyj0121/samsung/sogang/bias_frame/SN_20260327_123601
  /database2/iyj0121/samsung/sogang/bias_frame/SN_20260327_123606
  /database2/iyj0121/samsung/sogang/bias_frame/SN_20260327_123611
  /database2/iyj0121/samsung/sogang/bias_frame/SN_20260327_123618
  /database2/iyj0121/samsung/sogang/bias_frame/SN_20260327_123623
  /database2/iyj0121/samsung/sogang/bias_frame/SN_2026

dng용 bias frame 처리

In [58]:
import os
import glob
import numpy as np
import rawpy

# =========================
# 경로 설정
# =========================
dng_dir = "/database2/iyj0121/samsung/sogang/bias_raw_cap/400/8ms/"
save_dir = os.path.join(dng_dir, "mean_result")
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(save_dir, "bias_mean_150frames_u16.raw")

# =========================
# DNG 파일 수집
# =========================
all_dng_paths = sorted(glob.glob(os.path.join(dng_dir, "*.dng")))

print(f"Found DNG files: {len(all_dng_paths)}")
for p in all_dng_paths[:10]:
    print(f"  {p}")
if len(all_dng_paths) > 10:
    print("  ...")

expected_total = 150
if len(all_dng_paths) != expected_total:
    raise RuntimeError(f"Expected {expected_total} images, but got {len(all_dng_paths)}")

# =========================
# 첫 장 읽어서 shape 확인
# =========================
with rawpy.imread(all_dng_paths[0]) as raw:
    first_img = raw.raw_image.copy().astype(np.float64)

H, W = first_img.shape

print("\nFirst image info:")
print(f"shape: {first_img.shape}")
print(f"dtype: {first_img.dtype}")
print(f"min  : {first_img.min()}")
print(f"max  : {first_img.max()}")

# =========================
# Mean 계산
# =========================
acc = np.zeros((H, W), dtype=np.float64)

for i, path in enumerate(all_dng_paths, 1):
    with rawpy.imread(path) as raw:
        img = raw.raw_image.copy().astype(np.float64)

    if img.shape != (H, W):
        raise ValueError(
            f"Shape mismatch: {path}\n"
            f"  current shape = {img.shape}\n"
            f"  expected      = {(H, W)}"
        )

    acc += img

    if i % 10 == 0 or i == len(all_dng_paths):
        print(f"{i}/{len(all_dng_paths)} processed")

mean_img = acc / len(all_dng_paths)

# =========================
# uint16 저장
# =========================
mean_img = np.clip(mean_img, 0, 65535)
mean_img_u16 = mean_img.astype(np.uint16)

mean_img_u16.tofile(save_path)

print("\n=========================")
print(f"Mean RAW saved to: {save_path}")
print(f"dtype: {mean_img_u16.dtype}")
print(f"shape: {mean_img_u16.shape}")
print(f"min: {mean_img_u16.min()}")
print(f"max: {mean_img_u16.max()}")

# =========================
# 추가 디버깅용 통계
# =========================
print("\n[Debug statistics]")
print(f"single frame mean: {first_img.mean():.4f}")
print(f"single frame std : {first_img.std():.4f}")
print(f"mean image mean  : {mean_img.mean():.4f}")
print(f"mean image std   : {mean_img.std():.4f}")

Found DNG files: 150
  /database2/iyj0121/samsung/sogang/bias_raw_cap/400/8ms/20260403_154712_ISO400_8333us_001.dng
  /database2/iyj0121/samsung/sogang/bias_raw_cap/400/8ms/20260403_154712_ISO400_8333us_002.dng
  /database2/iyj0121/samsung/sogang/bias_raw_cap/400/8ms/20260403_154712_ISO400_8333us_003.dng
  /database2/iyj0121/samsung/sogang/bias_raw_cap/400/8ms/20260403_154712_ISO400_8333us_004.dng
  /database2/iyj0121/samsung/sogang/bias_raw_cap/400/8ms/20260403_154712_ISO400_8333us_005.dng
  /database2/iyj0121/samsung/sogang/bias_raw_cap/400/8ms/20260403_154712_ISO400_8333us_006.dng
  /database2/iyj0121/samsung/sogang/bias_raw_cap/400/8ms/20260403_154712_ISO400_8333us_007.dng
  /database2/iyj0121/samsung/sogang/bias_raw_cap/400/8ms/20260403_154712_ISO400_8333us_008.dng
  /database2/iyj0121/samsung/sogang/bias_raw_cap/400/8ms/20260403_154712_ISO400_8333us_009.dng
  /database2/iyj0121/samsung/sogang/bias_raw_cap/400/8ms/20260403_154712_ISO400_8333us_010.dng
  ...

First image info:
shap

center crop

In [59]:
import numpy as np
import os

# =========================
# 경로 설정
# =========================
input_path = "/database2/iyj0121/samsung/sogang/bias_raw_cap/400/8ms/mean_result/bias_mean_150frames_u16.raw"
output_path = "/database2/iyj0121/samsung/sogang/bias_raw_cap/400/8ms/mean_result/bias_mean_150frames_u16_crop_3000x4000.raw"

# =========================
# 원본 크기
# =========================
H = 3060
W = 4080

# =========================
# 타겟 크기
# =========================
target_H = 3000
target_W = 4000

# =========================
# RAW 로드
# =========================
raw = np.fromfile(input_path, dtype=np.uint16)

if raw.size != H * W:
    raise ValueError(f"Size mismatch: expected {H*W}, got {raw.size}")

raw = raw.reshape(H, W)

# =========================
# center crop 좌표 계산
# =========================
start_y = (H - target_H) // 2  # 30
start_x = (W - target_W) // 2  # 40

end_y = start_y + target_H
end_x = start_x + target_W

print(f"Cropping region:")
print(f"  y: {start_y} ~ {end_y}")
print(f"  x: {start_x} ~ {end_x}")

# =========================
# crop 수행
# =========================
cropped = raw[start_y:end_y, start_x:end_x]

# =========================
# 저장
# =========================
cropped.tofile(output_path)

print("\n=========================")
print(f"Saved cropped RAW to: {output_path}")
print(f"shape: {cropped.shape}")
print(f"min: {cropped.min()}")
print(f"max: {cropped.max()}")

Cropping region:
  y: 30 ~ 3030
  x: 40 ~ 4040

Saved cropped RAW to: /database2/iyj0121/samsung/sogang/bias_raw_cap/400/8ms/mean_result/bias_mean_150frames_u16_crop_3000x4000.raw
shape: (3000, 4000)
min: 60
max: 105


raw cap 기반 bias mean sub 코드

In [61]:
import os
import numpy as np

# =========================
# 경로 설정
# =========================
gt_path = "/database2/iyj0121/samsung/251205/scene_000001/SN_20251205_140252/SN_20251205_140252_01_4000x3000_unpacked_u16.raw"
bias_path = "/database2/iyj0121/samsung/sogang/bias_raw_cap/400/8ms/mean_result/bias_mean_150frames_u16_crop_3000x4000.raw"

save_dir = "/database2/iyj0121/samsung/sogang/bias_raw_cap/400/8ms/mean_result/"
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(save_dir, "scene_000027_gt_norm_minus_bias_norm_denorm_u16.raw")

# =========================
# RAW 설정
# =========================
H = 3000
W = 4000
WHITE_LEVEL = 4095.0
BLACK_LEVEL = 255.0


def load_raw16(path, h, w):
    data = np.fromfile(path, dtype=np.uint16)

    if data.size != h * w:
        raise ValueError(
            f"Size mismatch: {path}\n"
            f"  data size = {data.size}\n"
            f"  expected  = {h*w}"
        )

    return data.reshape(h, w)


def print_stats(name, img):
    print(f"\n[{name}]")
    print(f"shape: {img.shape}, dtype: {img.dtype}")
    print(f"min/max : {img.min():.6f} / {img.max():.6f}")
    print(f"mean/std: {img.mean():.6f} / {img.std():.6f}")


# =========================
# 데이터 로드
# =========================
gt = load_raw16(gt_path, H, W).astype(np.float64)
bias = load_raw16(bias_path, H, W).astype(np.float64)

print_stats("GT raw", gt)
print_stats("Bias raw", bias)

# =========================
# 고정 black level 사용
# 새 bias mean frame의 black level = 64
# =========================
black00 = BLACK_LEVEL
black01 = BLACK_LEVEL
black10 = BLACK_LEVEL
black11 = BLACK_LEVEL

print("\n[Fixed channel-wise black level]")
print(f"ch00: {black00}")
print(f"ch01: {black01}")
print(f"ch10: {black10}")
print(f"ch11: {black11}")

# =========================
# black map 생성
# =========================
black_map = np.empty((H, W), dtype=np.float64)
black_map[0::2, 0::2] = black00
black_map[0::2, 1::2] = black01
black_map[1::2, 0::2] = black10
black_map[1::2, 1::2] = black11

denom = WHITE_LEVEL - black_map
if np.any(denom <= 0):
    raise ValueError("Found non-positive denominator in normalization.")

# =========================
# 정규화
# =========================
gt_norm = (gt - black_map) / denom
bias_norm = (bias - 64) / (1023-64)

gt_norm = np.clip(gt_norm, 0.0, 1.0)
bias_norm = np.clip(bias_norm, 0.0, 1.0)

print_stats("GT norm", gt_norm)
print_stats("Bias norm", bias_norm)

# =========================
# norm domain subtraction
# =========================
result_norm = gt_norm - bias_norm
print_stats("Result norm before clip", result_norm)

# 음수 방지
result_norm_clip = np.clip(result_norm, 0.0, 1.0)
print_stats("Result norm after clip", result_norm_clip)

# =========================
# DN 단위로 복원
# result_norm = (gt - bias) / (white - black)
# 따라서 result_dn = result_norm * (white - black)
# 여기서는 black을 더하면 안 됨
# =========================
result_dn = result_norm_clip * denom + BLACK_LEVEL
result_dn = np.clip(result_dn, 0.0, WHITE_LEVEL)

print_stats("Result DN", result_dn)

# =========================
# uint16 저장
# =========================
result_u16 = np.round(result_dn).astype(np.uint16)
result_u16.tofile(save_path)

# =========================
# 로그 출력
# =========================
print("\n===================================")
print(f"Saved to: {save_path}")
print(f"dtype: {result_u16.dtype}, shape: {result_u16.shape}")
print(f"min: {result_u16.min()}, max: {result_u16.max()}")


[GT raw]
shape: (3000, 4000), dtype: float64
min/max : 228.000000 / 544.000000
mean/std: 258.387271 / 4.834635

[Bias raw]
shape: (3000, 4000), dtype: float64
min/max : 60.000000 / 105.000000
mean/std: 63.519629 / 0.560853

[Fixed channel-wise black level]
ch00: 255.0
ch01: 255.0
ch10: 255.0
ch11: 255.0

[GT norm]
shape: (3000, 4000), dtype: float64
min/max : 0.000000 / 0.075260
mean/std: 0.001046 / 0.001042

[Bias norm]
shape: (3000, 4000), dtype: float64
min/max : 0.000000 / 0.042753
mean/std: 0.000014 / 0.000132

[Result norm before clip]
shape: (3000, 4000), dtype: float64
min/max : -0.041451 / 0.075260
mean/std: 0.001032 / 0.001038

[Result norm after clip]
shape: (3000, 4000), dtype: float64
min/max : 0.000000 / 0.075260
mean/std: 0.001034 / 0.001034

[Result DN]
shape: (3000, 4000), dtype: float64
min/max : 255.000000 / 544.000000
mean/std: 258.971536 / 3.972262

Saved to: /database2/iyj0121/samsung/sogang/bias_raw_cap/400/8ms/mean_result/scene_000027_gt_norm_minus_bias_norm_de

고품질 GT 생성 코드

In [25]:
import os
import numpy as np

# =========================
# 경로 설정
# =========================
gt_path = "/database2/iyj0121/samsung/gt_mean/gt_mean_scene_000027/scene_000027_gt_mean_80frames_u16.raw"
bias_path = "/database2/iyj0121/samsung/sogang/bias_frame/mean_result/bias_mean_90frames_u16.raw"

save_dir = "/database2/iyj0121/samsung/sogang/bias_frame/mean_result/"
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(save_dir, "scene_000027_gt_norm_minus_bias_norm_denorm_u16.raw")

# =========================
# RAW 설정
# =========================
H = 3000
W = 4000
WHITE_LEVEL = 4095.0


def load_raw16(path, h, w):
    data = np.fromfile(path, dtype=np.uint16)

    if data.size != h * w:
        raise ValueError(
            f"Size mismatch: {path}\n"
            f"  data size = {data.size}\n"
            f"  expected  = {h*w}"
        )

    return data.reshape(h, w)


def print_stats(name, img):
    print(f"\n[{name}]")
    print(f"shape: {img.shape}, dtype: {img.dtype}")
    print(f"min/max : {img.min():.6f} / {img.max():.6f}")
    print(f"mean/std: {img.mean():.6f} / {img.std():.6f}")


# =========================
# 데이터 로드
# =========================
gt = load_raw16(gt_path, H, W).astype(np.float64)
bias = load_raw16(bias_path, H, W).astype(np.float64)

print_stats("GT raw", gt)
print_stats("Bias raw", bias)

# =========================
# Bayer 채널별 black level 추정
# bias median 사용
# =========================
black00 = np.median(bias[0::2, 0::2])
black01 = np.median(bias[0::2, 1::2])
black10 = np.median(bias[1::2, 0::2])
black11 = np.median(bias[1::2, 1::2])

print("\n[Channel-wise black level from bias median]")
print(f"ch00: {black00}")
print(f"ch01: {black01}")
print(f"ch10: {black10}")
print(f"ch11: {black11}")

# =========================
# black map 생성
# =========================
black_map = np.empty((H, W), dtype=np.float64)
black_map[0::2, 0::2] = black00
black_map[0::2, 1::2] = black01
black_map[1::2, 0::2] = black10
black_map[1::2, 1::2] = black11

denom = WHITE_LEVEL - black_map
if np.any(denom <= 0):
    raise ValueError("Found non-positive denominator in normalization.")

# =========================
# 정규화
# =========================
gt_norm = (gt - black_map) / denom
bias_norm = (bias - black_map) / denom

gt_norm = np.clip(gt_norm, 0.0, 1.0)
bias_norm = np.clip(bias_norm, 0.0, 1.0)

print_stats("GT norm", gt_norm)
print_stats("Bias norm", bias_norm)

# =========================
# norm domain subtraction
# =========================
result_norm = gt_norm - bias_norm
print_stats("Result norm before clip", result_norm)

# 음수 방지
result_norm_clip = np.clip(result_norm, 0.0, 1.0)
print_stats("Result norm after clip", result_norm_clip)

# =========================
# DN 단위로 복원
# result_norm = (gt - bias) / (white - black)
# 따라서 result_dn = result_norm * (white - black)
# 여기서는 black을 더하면 안 됨
# =========================
result_dn = result_norm_clip * denom + 256
result_dn = np.clip(result_dn, 0.0, WHITE_LEVEL)

print_stats("Result DN", result_dn)

# =========================
# uint16 저장
# =========================
result_u16 = np.round(result_dn).astype(np.uint16)
result_u16.tofile(save_path)

# =========================
# 로그 출력
# =========================
print("\n===================================")
print(f"Saved to: {save_path}")
print(f"dtype: {result_u16.dtype}, shape: {result_u16.shape}")
print(f"min: {result_u16.min()}, max: {result_u16.max()}")


[GT raw]
shape: (3000, 4000), dtype: float64
min/max : 223.000000 / 1668.000000
mean/std: 398.010974 / 121.078018

[Bias raw]
shape: (3000, 4000), dtype: float64
min/max : 214.000000 / 4092.000000
mean/std: 255.257930 / 7.805337

[Channel-wise black level from bias median]
ch00: 256.0
ch01: 255.0
ch10: 255.0
ch11: 256.0

[GT norm]
shape: (3000, 4000), dtype: float64
min/max : 0.000000 / 0.367804
mean/std: 0.037125 / 0.031492

[Bias norm]
shape: (3000, 4000), dtype: float64
min/max : 0.000000 / 0.999219
mean/std: 0.000572 / 0.001581

[Result norm before clip]
shape: (3000, 4000), dtype: float64
min/max : -0.953646 / 0.367804
mean/std: 0.036552 / 0.031507

[Result norm after clip]
shape: (3000, 4000), dtype: float64
min/max : 0.000000 / 0.367804
mean/std: 0.036558 / 0.031491

[Result DN]
shape: (3000, 4000), dtype: float64
min/max : 256.000000 / 1668.000000
mean/std: 396.361440 / 120.897449

Saved to: /database2/iyj0121/samsung/sogang/bias_frame/mean_result/scene_000027_gt_norm_minus_bi

디스파이크 버전

In [56]:
import os
import numpy as np

# =========================
# 경로 설정
# =========================
gt_path = "/database2/iyj0121/samsung/251205/scene_000001/mean_result/scene_000001_mean_240frames_u16.raw"
bias_path = "/database2/iyj0121/samsung/sogang/bias_frame/mean_result/bias_mean_90frames_u16.raw"

save_dir = "/database2/iyj0121/samsung/251205/scene_000001/mean_result/"
os.makedirs(save_dir, exist_ok=True)

save_bias_fixed_path = os.path.join(save_dir, "bias_mean_90frames_u16_despiked.raw")
save_result_path = os.path.join(save_dir, "scene_000027_gt_minus_bias_despiked_u16.raw")

# =========================
# RAW 설정
# =========================
H = 3000
W = 4000
WHITE_LEVEL = 4095.0
BLACK_OFFSET_FOR_SAVE = 256.0

# threshold 강도
# 값이 작을수록 더 많이 이상치로 잡음
ROBUST_SIGMA_K = 8.0


def load_raw16(path, h, w):
    data = np.fromfile(path, dtype=np.uint16)

    if data.size != h * w:
        raise ValueError(
            f"Size mismatch: {path}\n"
            f"  data size = {data.size}\n"
            f"  expected  = {h * w}"
        )

    return data.reshape(h, w)


def print_stats(name, img):
    print(f"\n[{name}]")
    print(f"shape: {img.shape}, dtype: {img.dtype}")
    print(f"min/max : {img.min():.6f} / {img.max():.6f}")
    print(f"mean/std: {img.mean():.6f} / {img.std():.6f}")


def bayer_channel_medians(img):
    return {
        "ch00": float(np.median(img[0::2, 0::2])),
        "ch01": float(np.median(img[0::2, 1::2])),
        "ch10": float(np.median(img[1::2, 0::2])),
        "ch11": float(np.median(img[1::2, 1::2])),
    }


def build_bayer_map(h, w, ch00, ch01, ch10, ch11):
    out = np.empty((h, w), dtype=np.float64)
    out[0::2, 0::2] = ch00
    out[0::2, 1::2] = ch01
    out[1::2, 0::2] = ch10
    out[1::2, 1::2] = ch11
    return out


def despike_bias_with_channel_median(bias, sigma_k=8.0):
    """
    bias에서 큰 이상치 픽셀을 robust threshold로 찾고,
    각 Bayer 채널 median으로 치환
    """
    bias_fixed = bias.copy()

    med = np.median(bias)
    mad = np.median(np.abs(bias - med))

    if mad == 0:
        robust_sigma = 1.0
    else:
        robust_sigma = 1.4826 * mad

    thr_high = med + sigma_k * robust_sigma

    print("\n[Robust outlier detection]")
    print(f"global median      : {med:.6f}")
    print(f"global MAD         : {mad:.6f}")
    print(f"robust sigma       : {robust_sigma:.6f}")
    print(f"high threshold     : {thr_high:.6f}")

    ch_meds = bayer_channel_medians(bias)
    print("\n[Channel medians for replacement]")
    for k, v in ch_meds.items():
        print(f"{k}: {v:.6f}")

    # 채널별 이상치 마스크
    mask00 = bias[0::2, 0::2] > thr_high
    mask01 = bias[0::2, 1::2] > thr_high
    mask10 = bias[1::2, 0::2] > thr_high
    mask11 = bias[1::2, 1::2] > thr_high

    print("\n[Outlier ratio by channel]")
    print(f"ch00: {mask00.mean() * 100:.6f}%")
    print(f"ch01: {mask01.mean() * 100:.6f}%")
    print(f"ch10: {mask10.mean() * 100:.6f}%")
    print(f"ch11: {mask11.mean() * 100:.6f}%")

    # 이상치를 채널 median으로 대체
    bias_fixed[0::2, 0::2][mask00] = ch_meds["ch00"]
    bias_fixed[0::2, 1::2][mask01] = ch_meds["ch01"]
    bias_fixed[1::2, 0::2][mask10] = ch_meds["ch10"]
    bias_fixed[1::2, 1::2][mask11] = ch_meds["ch11"]

    total_mask = np.zeros_like(bias, dtype=bool)
    total_mask[0::2, 0::2] = mask00
    total_mask[0::2, 1::2] = mask01
    total_mask[1::2, 0::2] = mask10
    total_mask[1::2, 1::2] = mask11

    print(f"\nTotal outlier ratio: {total_mask.mean() * 100:.6f}%")

    return bias_fixed, total_mask


# =========================
# 데이터 로드
# =========================
gt = load_raw16(gt_path, H, W).astype(np.float64)
bias = load_raw16(bias_path, H, W).astype(np.float64)

print_stats("GT raw", gt)
print_stats("Bias raw (original)", bias)

# =========================
# bias 이상치 제거
# =========================
bias_fixed, outlier_mask = despike_bias_with_channel_median(
    bias, sigma_k=ROBUST_SIGMA_K
)

print_stats("Bias raw (despiked)", bias_fixed)

# 저장: despiked bias map
bias_fixed_u16 = np.round(np.clip(bias_fixed, 0, WHITE_LEVEL)).astype(np.uint16)
bias_fixed_u16.tofile(save_bias_fixed_path)
print(f"\nSaved despiked bias to: {save_bias_fixed_path}")

# =========================
# Bayer 채널별 black level 추정
# despiked bias의 median 사용
# =========================
ch_meds_fixed = bayer_channel_medians(bias_fixed)

black00 = ch_meds_fixed["ch00"]
black01 = ch_meds_fixed["ch01"]
black10 = ch_meds_fixed["ch10"]
black11 = ch_meds_fixed["ch11"]

print("\n[Black levels from despiked bias medians]")
print(f"ch00: {black00:.6f}")
print(f"ch01: {black01:.6f}")
print(f"ch10: {black10:.6f}")
print(f"ch11: {black11:.6f}")

black_map = build_bayer_map(H, W, black00, black01, black10, black11)

denom = WHITE_LEVEL - black_map
if np.any(denom <= 0):
    raise ValueError("Found non-positive denominator in normalization.")

# =========================
# 정규화
# =========================
gt_norm = (gt - black_map) / denom
bias_norm = (bias_fixed - black_map) / denom

gt_norm = np.clip(gt_norm, 0.0, 1.0)
bias_norm = np.clip(bias_norm, 0.0, 1.0)

print_stats("GT norm", gt_norm)
print_stats("Bias norm (despiked)", bias_norm)

# =========================
# subtraction in normalized domain
# =========================
result_norm = gt_norm - bias_norm
print_stats("Result norm before clip", result_norm)

result_norm_clip = np.clip(result_norm, 0.0, 1.0)
print_stats("Result norm after clip", result_norm_clip)

# =========================
# DN 단위 복원
# 네가 현재 결과가 정상적이라고 한 방식 유지
# =========================
result_dn = result_norm_clip * denom + BLACK_OFFSET_FOR_SAVE
result_dn = np.clip(result_dn, 0.0, WHITE_LEVEL)

print_stats("Result DN", result_dn)

# =========================
# 저장
# =========================
result_u16 = np.round(result_dn).astype(np.uint16)
result_u16.tofile(save_result_path)

print("\n===================================")
print(f"Saved final result to: {save_result_path}")
print(f"dtype: {result_u16.dtype}")
print(f"shape: {result_u16.shape}")
print(f"min: {result_u16.min()}")
print(f"max: {result_u16.max()}")

# =========================
# 추가 디버깅
# =========================
print("\n[Debug]")
print(f"Pixels replaced in bias: {outlier_mask.sum()} / {outlier_mask.size}")
print(f"Replaced ratio         : {outlier_mask.mean() * 100:.6f}%")


[GT raw]
shape: (3000, 4000), dtype: float64
min/max : 225.000000 / 4092.000000
mean/std: 335.135009 / 70.379030

[Bias raw (original)]
shape: (3000, 4000), dtype: float64
min/max : 214.000000 / 4092.000000
mean/std: 255.257930 / 7.805337

[Robust outlier detection]
global median      : 255.000000
global MAD         : 4.000000
robust sigma       : 5.930400
high threshold     : 302.443200

[Channel medians for replacement]
ch00: 256.000000
ch01: 255.000000
ch10: 255.000000
ch11: 256.000000

[Outlier ratio by channel]
ch00: 0.026033%
ch01: 0.026633%
ch10: 0.026067%
ch11: 0.027467%

Total outlier ratio: 0.026550%

[Bias raw (despiked)]
shape: (3000, 4000), dtype: float64
min/max : 214.000000 / 302.000000
mean/std: 255.195566 / 5.857557

Saved despiked bias to: /database2/iyj0121/samsung/251205/scene_000001/mean_result/bias_mean_90frames_u16_despiked.raw

[Black levels from despiked bias medians]
ch00: 256.000000
ch01: 255.000000
ch10: 255.000000
ch11: 256.000000

[GT norm]
shape: (3000, 

노 디스파이크

In [53]:
import os
import numpy as np

# =========================
# 경로 설정
# =========================
gt_path = "/database2/iyj0121/samsung/251205/scene_000001/mean_result/scene_000001_mean_240frames_u16.raw"
bias_path = "/database2/iyj0121/samsung/sogang/bias_frame/mean_result/bias_mean_90frames_u16.raw"

save_dir = "/database2/iyj0121/samsung/251205/scene_000001/mean_result/"
os.makedirs(save_dir, exist_ok=True)

save_result_path = os.path.join(save_dir, "scene_000001_gt_minus_bias_u16.raw")

# =========================
# RAW 설정
# =========================
H = 3000
W = 4000
WHITE_LEVEL = 4095.0
BLACK_OFFSET_FOR_SAVE = 256.0


def load_raw16(path, h, w):
    data = np.fromfile(path, dtype=np.uint16)

    if data.size != h * w:
        raise ValueError(
            f"Size mismatch: {path}\n"
            f"  data size = {data.size}\n"
            f"  expected  = {h * w}"
        )

    return data.reshape(h, w)


def print_stats(name, img):
    print(f"\n[{name}]")
    print(f"shape: {img.shape}, dtype: {img.dtype}")
    print(f"min/max : {img.min():.6f} / {img.max():.6f}")
    print(f"mean/std: {img.mean():.6f} / {img.std():.6f}")


def bayer_channel_medians(img):
    return {
        "ch00": float(np.median(img[0::2, 0::2])),
        "ch01": float(np.median(img[0::2, 1::2])),
        "ch10": float(np.median(img[1::2, 0::2])),
        "ch11": float(np.median(img[1::2, 1::2])),
    }


def build_bayer_map(h, w, ch00, ch01, ch10, ch11):
    out = np.empty((h, w), dtype=np.float64)
    out[0::2, 0::2] = ch00
    out[0::2, 1::2] = ch01
    out[1::2, 0::2] = ch10
    out[1::2, 1::2] = ch11
    return out


# =========================
# 데이터 로드
# =========================
gt = load_raw16(gt_path, H, W).astype(np.float64)
bias = load_raw16(bias_path, H, W).astype(np.float64)

print_stats("GT raw", gt)
print_stats("Bias raw", bias)

# =========================
# Bayer 채널별 black level 추정 (bias 기준)
# =========================
ch_meds = bayer_channel_medians(bias)

black00 = ch_meds["ch00"]
black01 = ch_meds["ch01"]
black10 = ch_meds["ch10"]
black11 = ch_meds["ch11"]

print("\n[Black levels from bias]")
print(f"ch00: {black00:.6f}")
print(f"ch01: {black01:.6f}")
print(f"ch10: {black10:.6f}")
print(f"ch11: {black11:.6f}")

black_map = build_bayer_map(H, W, black00, black01, black10, black11)

denom = WHITE_LEVEL - black_map
if np.any(denom <= 0):
    raise ValueError("Found non-positive denominator in normalization.")

# =========================
# 정규화
# =========================
gt_norm = (gt - black_map) / denom
bias_norm = (bias - black_map) / denom

gt_norm = np.clip(gt_norm, 0.0, 1.0)
bias_norm = np.clip(bias_norm, 0.0, 1.0)

print_stats("GT norm", gt_norm)
print_stats("Bias norm", bias_norm)

# =========================
# subtraction
# =========================
result_norm = gt_norm - bias_norm
print_stats("Result norm before clip", result_norm)

result_norm_clip = np.clip(result_norm, 0.0, 1.0)
print_stats("Result norm after clip", result_norm_clip)

# =========================
# DN 복원
# =========================
result_dn = result_norm_clip * denom + BLACK_OFFSET_FOR_SAVE
result_dn = np.clip(result_dn, 0.0, WHITE_LEVEL)

print_stats("Result DN", result_dn)

# =========================
# 저장
# =========================
result_u16 = np.round(result_dn).astype(np.uint16)
result_u16.tofile(save_result_path)

print("\n===================================")
print(f"Saved final result to: {save_result_path}")
print(f"dtype: {result_u16.dtype}")
print(f"shape: {result_u16.shape}")
print(f"min: {result_u16.min()}")
print(f"max: {result_u16.max()}")


[GT raw]
shape: (3000, 4000), dtype: float64
min/max : 225.000000 / 4092.000000
mean/std: 335.135009 / 70.379030

[Bias raw]
shape: (3000, 4000), dtype: float64
min/max : 214.000000 / 4092.000000
mean/std: 255.257930 / 7.805337

[Black levels from bias]
ch00: 256.000000
ch01: 255.000000
ch10: 255.000000
ch11: 256.000000

[GT norm]
shape: (3000, 4000), dtype: float64
min/max : 0.000000 / 0.999219
mean/std: 0.020753 / 0.018277

[Bias norm]
shape: (3000, 4000), dtype: float64
min/max : 0.000000 / 0.999219
mean/std: 0.000572 / 0.001581

[Result norm before clip]
shape: (3000, 4000), dtype: float64
min/max : -0.980729 / 0.999219
mean/std: 0.020181 / 0.018255

[Result norm after clip]
shape: (3000, 4000), dtype: float64
min/max : 0.000000 / 0.999219
mean/std: 0.020186 / 0.018239

[Result DN]
shape: (3000, 4000), dtype: float64
min/max : 256.000000 / 4092.000000
mean/std: 333.501622 / 70.020692

Saved final result to: /database2/iyj0121/samsung/251205/scene_000001/mean_result/scene_000001_gt

In [20]:
import os
import numpy as np

# =========================
# 공통 설정
# =========================
BASE_DIR = "/database2/iyj0121/samsung"
GT_BASE_DIR = os.path.join(BASE_DIR, "gt_mean")
SAVE_BASE_DIR = os.path.join(BASE_DIR, "gt_processed")
BIAS_PATH = "/database2/iyj0121/samsung/sogang/bias_frame/mean_result/bias_mean_90frames_u16.raw"

os.makedirs(SAVE_BASE_DIR, exist_ok=True)

H = 3000
W = 4000
WHITE_LEVEL = 4095.0
BLACK_OFFSET_FOR_SAVE = 256.0
ROBUST_SIGMA_K = 8.0


# =========================
# 공통 함수
# =========================
def load_raw16(path):
    data = np.fromfile(path, dtype=np.uint16)
    if data.size != H * W:
        raise ValueError(f"Size mismatch: {path}")
    return data.reshape(H, W).astype(np.float64)


def bayer_channel_medians(img):
    return {
        "ch00": np.median(img[0::2, 0::2]),
        "ch01": np.median(img[0::2, 1::2]),
        "ch10": np.median(img[1::2, 0::2]),
        "ch11": np.median(img[1::2, 1::2]),
    }


def build_bayer_map(ch):
    out = np.empty((H, W), dtype=np.float64)
    out[0::2, 0::2] = ch["ch00"]
    out[0::2, 1::2] = ch["ch01"]
    out[1::2, 0::2] = ch["ch10"]
    out[1::2, 1::2] = ch["ch11"]
    return out


def despike_bias(bias):
    med = np.median(bias)
    mad = np.median(np.abs(bias - med))
    robust_sigma = 1.4826 * mad if mad > 0 else 1.0

    thr = med + ROBUST_SIGMA_K * robust_sigma

    ch_meds = bayer_channel_medians(bias)

    bias_fixed = bias.copy()

    # 채널별 마스크
    mask00 = bias[0::2, 0::2] > thr
    mask01 = bias[0::2, 1::2] > thr
    mask10 = bias[1::2, 0::2] > thr
    mask11 = bias[1::2, 1::2] > thr

    bias_fixed[0::2, 0::2][mask00] = ch_meds["ch00"]
    bias_fixed[0::2, 1::2][mask01] = ch_meds["ch01"]
    bias_fixed[1::2, 0::2][mask10] = ch_meds["ch10"]
    bias_fixed[1::2, 1::2][mask11] = ch_meds["ch11"]

    total_mask = np.zeros_like(bias, dtype=bool)
    total_mask[0::2, 0::2] = mask00
    total_mask[0::2, 1::2] = mask01
    total_mask[1::2, 0::2] = mask10
    total_mask[1::2, 1::2] = mask11

    print(f"[Bias] outlier ratio: {total_mask.mean() * 100:.6f}%")

    return bias_fixed


# =========================
# bias 한 번만 전처리
# =========================
print("Loading bias...")
bias = load_raw16(BIAS_PATH)
bias_fixed = despike_bias(bias)

ch_black = bayer_channel_medians(bias_fixed)
black_map = build_bayer_map(ch_black)
denom = WHITE_LEVEL - black_map

print("Bias preprocessing done.")

# =========================
# 모든 scene 처리
# =========================
for scene_id in range(1, 28):
    scene_name = f"scene_{scene_id:06d}"

    gt_path = os.path.join(
        GT_BASE_DIR,
        f"gt_mean_{scene_name}",
        f"{scene_name}_gt_mean_80frames_u16.raw"
    )

    if not os.path.exists(gt_path):
        print(f"[Skip] {gt_path} not found")
        continue

    print(f"\nProcessing {scene_name}...")

    # =========================
    # GT load
    # =========================
    gt = load_raw16(gt_path)

    # =========================
    # normalization
    # =========================
    gt_norm = (gt - black_map) / denom
    bias_norm = (bias_fixed - black_map) / denom

    gt_norm = np.clip(gt_norm, 0.0, 1.0)
    bias_norm = np.clip(bias_norm, 0.0, 1.0)

    # =========================
    # subtraction
    # =========================
    result_norm = gt_norm - bias_norm
    result_norm = np.clip(result_norm, 0.0, 1.0)

    # =========================
    # DN 복원 (네가 검증한 방식)
    # =========================
    result_dn = result_norm * denom + BLACK_OFFSET_FOR_SAVE
    result_dn = np.clip(result_dn, 0.0, WHITE_LEVEL)

    # =========================
    # 저장
    # =========================
    save_path = os.path.join(
        SAVE_BASE_DIR,
        f"{scene_name}_processed_u16.raw"
    )

    result_u16 = np.round(result_dn).astype(np.uint16)
    result_u16.tofile(save_path)

    print(f"[Saved] {save_path}")

print("\nAll scenes processed.")

Loading bias...
[Bias] outlier ratio: 0.026550%
Bias preprocessing done.

Processing scene_000001...
[Saved] /database2/iyj0121/samsung/gt_processed/scene_000001_processed_u16.raw

Processing scene_000002...
[Saved] /database2/iyj0121/samsung/gt_processed/scene_000002_processed_u16.raw

Processing scene_000003...
[Saved] /database2/iyj0121/samsung/gt_processed/scene_000003_processed_u16.raw

Processing scene_000004...
[Saved] /database2/iyj0121/samsung/gt_processed/scene_000004_processed_u16.raw

Processing scene_000005...
[Saved] /database2/iyj0121/samsung/gt_processed/scene_000005_processed_u16.raw

Processing scene_000006...
[Saved] /database2/iyj0121/samsung/gt_processed/scene_000006_processed_u16.raw

Processing scene_000007...
[Saved] /database2/iyj0121/samsung/gt_processed/scene_000007_processed_u16.raw

Processing scene_000008...
[Saved] /database2/iyj0121/samsung/gt_processed/scene_000008_processed_u16.raw

Processing scene_000009...
[Saved] /database2/iyj0121/samsung/gt_proce

그냥 norm할 때 256 빼는 코드

In [1]:
import os
import numpy as np

# =========================
# 공통 설정
# =========================
BASE_DIR = "/database2/iyj0121/samsung"
GT_BASE_DIR = os.path.join(BASE_DIR, "gt_mean")
SAVE_BASE_DIR = os.path.join(BASE_DIR, "gt_processed_black")
BIAS_PATH = "/database2/iyj0121/samsung/sogang/bias_frame/mean_result/bias_mean_90frames_u16.raw"

os.makedirs(SAVE_BASE_DIR, exist_ok=True)

H = 3000
W = 4000
WHITE_LEVEL = 4095.0
BLACK_LEVEL = 256.0
ROBUST_SIGMA_K = 8.0


# =========================
# 공통 함수
# =========================
def load_raw16(path):
    data = np.fromfile(path, dtype=np.uint16)
    if data.size != H * W:
        raise ValueError(f"Size mismatch: {path}")
    return data.reshape(H, W).astype(np.float64)


def bayer_channel_medians(img):
    return {
        "ch00": np.median(img[0::2, 0::2]),
        "ch01": np.median(img[0::2, 1::2]),
        "ch10": np.median(img[1::2, 0::2]),
        "ch11": np.median(img[1::2, 1::2]),
    }


def despike_bias(bias):
    med = np.median(bias)
    mad = np.median(np.abs(bias - med))
    robust_sigma = 1.4826 * mad if mad > 0 else 1.0

    thr = med + ROBUST_SIGMA_K * robust_sigma

    ch_meds = bayer_channel_medians(bias)

    bias_fixed = bias.copy()

    mask00 = bias[0::2, 0::2] > thr
    mask01 = bias[0::2, 1::2] > thr
    mask10 = bias[1::2, 0::2] > thr
    mask11 = bias[1::2, 1::2] > thr

    bias_fixed[0::2, 0::2][mask00] = ch_meds["ch00"]
    bias_fixed[0::2, 1::2][mask01] = ch_meds["ch01"]
    bias_fixed[1::2, 0::2][mask10] = ch_meds["ch10"]
    bias_fixed[1::2, 1::2][mask11] = ch_meds["ch11"]

    total_mask = np.zeros_like(bias, dtype=bool)
    total_mask[0::2, 0::2] = mask00
    total_mask[0::2, 1::2] = mask01
    total_mask[1::2, 0::2] = mask10
    total_mask[1::2, 1::2] = mask11

    print(f"[Bias] outlier ratio: {total_mask.mean() * 100:.6f}%")

    return bias_fixed


# =========================
# bias 한 번만 전처리
# =========================
print("Loading bias...")
bias = load_raw16(BIAS_PATH)
bias_fixed = despike_bias(bias)

print("Bias preprocessing done.")

# =========================
# 모든 scene 처리
# =========================
for scene_id in range(1, 28):
    scene_name = f"scene_{scene_id:06d}"

    gt_path = os.path.join(
        GT_BASE_DIR,
        f"gt_mean_{scene_name}",
        f"{scene_name}_gt_mean_80frames_u16.raw"
    )

    if not os.path.exists(gt_path):
        print(f"[Skip] {gt_path} not found")
        continue

    print(f"\nProcessing {scene_name}...")

    # =========================
    # GT load
    # =========================
    gt = load_raw16(gt_path)

    # =========================
    # normalization (black = 256)
    # =========================
    denom = (WHITE_LEVEL - BLACK_LEVEL)

    gt_norm = (gt - BLACK_LEVEL) / denom
    bias_norm = (bias_fixed - BLACK_LEVEL) / denom

    gt_norm = np.clip(gt_norm, 0.0, 1.0)
    bias_norm = np.clip(bias_norm, 0.0, 1.0)

    # =========================
    # subtraction
    # =========================
    result_norm = gt_norm - bias_norm
    result_norm = np.clip(result_norm, 0.0, 1.0)

    # =========================
    # DN 복원
    # =========================
    result_dn = result_norm * denom + BLACK_LEVEL
    result_dn = np.clip(result_dn, 0.0, WHITE_LEVEL)

    # =========================
    # 저장
    # =========================
    save_path = os.path.join(
        SAVE_BASE_DIR,
        f"{scene_name}_processed_u16.raw"
    )

    result_u16 = np.round(result_dn).astype(np.uint16)
    result_u16.tofile(save_path)

    print(f"[Saved] {save_path}")

print("\nAll scenes processed.")

Loading bias...
[Bias] outlier ratio: 0.026550%
Bias preprocessing done.

Processing scene_000001...
[Saved] /database2/iyj0121/samsung/gt_processed_black/scene_000001_processed_u16.raw

Processing scene_000002...
[Saved] /database2/iyj0121/samsung/gt_processed_black/scene_000002_processed_u16.raw

Processing scene_000003...
[Saved] /database2/iyj0121/samsung/gt_processed_black/scene_000003_processed_u16.raw

Processing scene_000004...
[Saved] /database2/iyj0121/samsung/gt_processed_black/scene_000004_processed_u16.raw

Processing scene_000005...
[Saved] /database2/iyj0121/samsung/gt_processed_black/scene_000005_processed_u16.raw

Processing scene_000006...
[Saved] /database2/iyj0121/samsung/gt_processed_black/scene_000006_processed_u16.raw

Processing scene_000007...
[Saved] /database2/iyj0121/samsung/gt_processed_black/scene_000007_processed_u16.raw

Processing scene_000008...
[Saved] /database2/iyj0121/samsung/gt_processed_black/scene_000008_processed_u16.raw

Processing scene_00000

In [23]:
import os
import numpy as np

# =========================
# 공통 경로 설정
# =========================
GT_BASE_DIR = "/database2/iyj0121/samsung/251205_gt_mean"
BIAS_PATH = "/database2/iyj0121/samsung/sogang/bias_frame/mean_result/bias_mean_90frames_u16.raw"
SAVE_BASE_DIR = "/database2/iyj0121/samsung/251205_gt_mean_bias_process"

os.makedirs(SAVE_BASE_DIR, exist_ok=True)

# =========================
# RAW 설정
# =========================
H = 3000
W = 4000
WHITE_LEVEL = 4095.0
BLACK_OFFSET_FOR_SAVE = 256.0
ROBUST_SIGMA_K = 8.0

# 처리할 scene 범위
SCENE_START = 1
SCENE_END = 19


# =========================
# 유틸 함수
# =========================
def load_raw16(path, h, w):
    data = np.fromfile(path, dtype=np.uint16)

    if data.size != h * w:
        raise ValueError(
            f"Size mismatch: {path}\n"
            f"  data size = {data.size}\n"
            f"  expected  = {h*w}"
        )

    return data.reshape(h, w).astype(np.float64)


def print_stats(name, img):
    print(f"\n[{name}]")
    print(f"shape: {img.shape}, dtype: {img.dtype}")
    print(f"min/max : {img.min():.6f} / {img.max():.6f}")
    print(f"mean/std: {img.mean():.6f} / {img.std():.6f}")


def bayer_channel_medians(img):
    return {
        "ch00": float(np.median(img[0::2, 0::2])),
        "ch01": float(np.median(img[0::2, 1::2])),
        "ch10": float(np.median(img[1::2, 0::2])),
        "ch11": float(np.median(img[1::2, 1::2])),
    }


def build_bayer_map(h, w, ch00, ch01, ch10, ch11):
    out = np.empty((h, w), dtype=np.float64)
    out[0::2, 0::2] = ch00
    out[0::2, 1::2] = ch01
    out[1::2, 0::2] = ch10
    out[1::2, 1::2] = ch11
    return out


def despike_bias_with_channel_median(bias, sigma_k=8.0):
    """
    bias에서 큰 이상치 픽셀을 robust threshold로 찾고,
    각 Bayer 채널 median으로 치환
    """
    bias_fixed = bias.copy()

    med = np.median(bias)
    mad = np.median(np.abs(bias - med))

    if mad == 0:
        robust_sigma = 1.0
    else:
        robust_sigma = 1.4826 * mad

    thr_high = med + sigma_k * robust_sigma

    print("\n[Robust outlier detection]")
    print(f"global median      : {med:.6f}")
    print(f"global MAD         : {mad:.6f}")
    print(f"robust sigma       : {robust_sigma:.6f}")
    print(f"high threshold     : {thr_high:.6f}")

    ch_meds = bayer_channel_medians(bias)

    print("\n[Channel medians for replacement]")
    for k, v in ch_meds.items():
        print(f"{k}: {v:.6f}")

    # 채널별 이상치 마스크
    mask00 = bias[0::2, 0::2] > thr_high
    mask01 = bias[0::2, 1::2] > thr_high
    mask10 = bias[1::2, 0::2] > thr_high
    mask11 = bias[1::2, 1::2] > thr_high

    print("\n[Outlier ratio by channel]")
    print(f"ch00: {mask00.mean() * 100:.6f}%")
    print(f"ch01: {mask01.mean() * 100:.6f}%")
    print(f"ch10: {mask10.mean() * 100:.6f}%")
    print(f"ch11: {mask11.mean() * 100:.6f}%")

    # 이상치 치환
    bias_fixed[0::2, 0::2][mask00] = ch_meds["ch00"]
    bias_fixed[0::2, 1::2][mask01] = ch_meds["ch01"]
    bias_fixed[1::2, 0::2][mask10] = ch_meds["ch10"]
    bias_fixed[1::2, 1::2][mask11] = ch_meds["ch11"]

    total_mask = np.zeros_like(bias, dtype=bool)
    total_mask[0::2, 0::2] = mask00
    total_mask[0::2, 1::2] = mask01
    total_mask[1::2, 0::2] = mask10
    total_mask[1::2, 1::2] = mask11

    print(f"\nTotal outlier ratio: {total_mask.mean() * 100:.6f}%")

    return bias_fixed, total_mask


# =========================
# bias 1회 전처리
# =========================
print("Loading bias...")
bias = load_raw16(BIAS_PATH, H, W)
print_stats("Bias raw (original)", bias)

bias_fixed, outlier_mask = despike_bias_with_channel_median(
    bias, sigma_k=ROBUST_SIGMA_K
)
print_stats("Bias raw (despiked)", bias_fixed)

# despiked bias 저장
bias_fixed_save_path = os.path.join(SAVE_BASE_DIR, "bias_mean_90frames_despiked_u16.raw")
bias_fixed_u16 = np.round(np.clip(bias_fixed, 0, WHITE_LEVEL)).astype(np.uint16)
bias_fixed_u16.tofile(bias_fixed_save_path)
print(f"\nSaved despiked bias to: {bias_fixed_save_path}")

# channel-wise black map 생성
ch_meds_fixed = bayer_channel_medians(bias_fixed)

black00 = ch_meds_fixed["ch00"]
black01 = ch_meds_fixed["ch01"]
black10 = ch_meds_fixed["ch10"]
black11 = ch_meds_fixed["ch11"]

print("\n[Black levels from despiked bias medians]")
print(f"ch00: {black00:.6f}")
print(f"ch01: {black01:.6f}")
print(f"ch10: {black10:.6f}")
print(f"ch11: {black11:.6f}")

black_map = build_bayer_map(H, W, black00, black01, black10, black11)

denom = WHITE_LEVEL - black_map
if np.any(denom <= 0):
    raise ValueError("Found non-positive denominator in normalization.")

print("\nBias preprocessing done.")
print(f"Pixels replaced in bias: {outlier_mask.sum()} / {outlier_mask.size}")
print(f"Replaced ratio         : {outlier_mask.mean() * 100:.6f}%")

# 미리 bias_norm 계산
bias_norm = (bias_fixed - black_map) / denom
bias_norm = np.clip(bias_norm, 0.0, 1.0)
print_stats("Bias norm (despiked)", bias_norm)


# =========================
# scene 전체 처리
# =========================
for scene_id in range(SCENE_START, SCENE_END + 1):
    scene_name = f"scene_{scene_id:06d}"
    gt_dir = os.path.join(GT_BASE_DIR, f"ev_gt_mean_{scene_name}")
    gt_name = f"{scene_name}_evm6_gt_mean_30frames_u16.raw"
    gt_path = os.path.join(gt_dir, gt_name)

    if not os.path.exists(gt_path):
        print(f"\n[Skip] not found: {gt_path}")
        continue

    print("\n" + "=" * 60)
    print(f"Processing {scene_name}")
    print("=" * 60)

    # GT load
    gt = load_raw16(gt_path, H, W)
    print_stats("GT raw", gt)

    # normalization
    gt_norm = (gt - black_map) / denom
    gt_norm = np.clip(gt_norm, 0.0, 1.0)
    print_stats("GT norm", gt_norm)

    # subtraction in normalized domain
    result_norm = gt_norm - bias_norm
    print_stats("Result norm before clip", result_norm)

    result_norm_clip = np.clip(result_norm, 0.0, 1.0)
    print_stats("Result norm after clip", result_norm_clip)

    # DN 복원
    result_dn = result_norm_clip * denom + BLACK_OFFSET_FOR_SAVE
    result_dn = np.clip(result_dn, 0.0, WHITE_LEVEL)
    print_stats("Result DN", result_dn)

    # 저장
    save_name = f"{scene_name}_evm6_gt_mean_30frames_bias_processed_u16.raw"
    save_path = os.path.join(SAVE_BASE_DIR, save_name)

    result_u16 = np.round(result_dn).astype(np.uint16)
    result_u16.tofile(save_path)

    print(f"[Saved] {save_path}")

print("\nAll scenes processed.")

Loading bias...

[Bias raw (original)]
shape: (3000, 4000), dtype: float64
min/max : 214.000000 / 4092.000000
mean/std: 255.257930 / 7.805337

[Robust outlier detection]
global median      : 255.000000
global MAD         : 4.000000
robust sigma       : 5.930400
high threshold     : 302.443200

[Channel medians for replacement]
ch00: 256.000000
ch01: 255.000000
ch10: 255.000000
ch11: 256.000000

[Outlier ratio by channel]
ch00: 0.026033%
ch01: 0.026633%
ch10: 0.026067%
ch11: 0.027467%

Total outlier ratio: 0.026550%

[Bias raw (despiked)]
shape: (3000, 4000), dtype: float64
min/max : 214.000000 / 302.000000
mean/std: 255.195566 / 5.857557

Saved despiked bias to: /database2/iyj0121/samsung/251205_gt_mean_bias_process/bias_mean_90frames_despiked_u16.raw

[Black levels from despiked bias medians]
ch00: 256.000000
ch01: 255.000000
ch10: 255.000000
ch11: 256.000000

Bias preprocessing done.
Pixels replaced in bias: 3186 / 12000000
Replaced ratio         : 0.026550%

[Bias norm (despiked)]
s

201205 scene1 GT 생성

In [52]:
import os
import glob
import numpy as np

# =========================
# 경로 설정
# =========================
root_pattern = "/database2/iyj0121/samsung/251205/scene_000001/SN_20251205_*"
save_dir = "/database2/iyj0121/samsung/251205/scene_000001/mean_result"
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(save_dir, "scene_000001_mean_240frames_u16.raw")

# =========================
# RAW 설정
# =========================
H = 3000
W = 4000

# =========================
# RAW12 packed 로드 함수
# =========================
def load_raw12_packed(path, h, w):
    """
    MIPI RAW12 packed 형식 로드
    2 pixels -> 3 bytes

    저장 방식:
      byte0 = p0[11:4]
      byte1 = p1[11:4]
      byte2 = (p1[3:0] << 4) | p0[3:0]

    복원:
      p0 = (byte0 << 4) | (byte2 & 0x0F)
      p1 = (byte1 << 4) | (byte2 >> 4)
    """
    expected_bytes = h * w * 3 // 2
    file_size = os.path.getsize(path)

    if file_size != expected_bytes:
        raise ValueError(
            f"Size mismatch: {path}\n"
            f"  file_size(bytes) = {file_size}\n"
            f"  expected(bytes)  = {expected_bytes}"
        )

    data = np.fromfile(path, dtype=np.uint8)

    if data.size % 3 != 0:
        raise ValueError(f"Packed RAW12 data size is not divisible by 3: {path}")

    data = data.reshape(-1, 3)

    b0 = data[:, 0].astype(np.uint16)
    b1 = data[:, 1].astype(np.uint16)
    b2 = data[:, 2].astype(np.uint16)

    p0 = (b0 << 4) | (b2 & 0x0F)
    p1 = (b1 << 4) | (b2 >> 4)

    raw = np.empty(p0.size + p1.size, dtype=np.uint16)
    raw[0::2] = p0
    raw[1::2] = p1

    if raw.size != h * w:
        raise ValueError(
            f"Unpacked pixel count mismatch: {path}\n"
            f"  unpacked pixels = {raw.size}\n"
            f"  expected pixels = {h * w}"
        )

    return raw.reshape(h, w)

# =========================
# 디렉토리 수집
# =========================
all_dirs = sorted(glob.glob(root_pattern))
all_dirs = [d for d in all_dirs if os.path.isdir(d)]

print(f"Found directories: {len(all_dirs)}")
if len(all_dirs) != 30:
    print(f"[Warning] Expected 30 directories, but found {len(all_dirs)}")

# =========================
# 파일 수집
# =========================
target_x = ["00", "04", "05", "06", "07", "08", "09", "10"]
all_raw_paths = []

for d in all_dirs:
    for x in target_x:
        pattern = os.path.join(d, f"SN_20251205_*_{x}_4000x3000.raw")
        files = sorted(glob.glob(pattern))

        if len(files) != 1:
            print(f"[Warning] {d} x={x} -> found {len(files)} files")
            for f in files:
                print(f"   - {os.path.basename(f)}")

        all_raw_paths.extend(files)

print(f"\nTotal selected RAW files: {len(all_raw_paths)}")

expected_total = 218
if len(all_raw_paths) != expected_total:
    raise RuntimeError(f"Expected {expected_total} images, but got {len(all_raw_paths)}")

# =========================
# Mean 계산
# =========================
acc = np.zeros((H, W), dtype=np.float64)

for i, path in enumerate(all_raw_paths, 1):
    img = load_raw12_packed(path, H, W).astype(np.float64)
    acc += img

    if i % 20 == 0 or i == len(all_raw_paths):
        print(f"{i}/{len(all_raw_paths)} processed")

mean_img = acc / len(all_raw_paths)

# =========================
# 12bit 범위 클리핑 후 저장
# =========================
mean_img = np.clip(mean_img, 0, 4095)
mean_img_u16 = mean_img.astype(np.uint16)

mean_img_u16.tofile(save_path)

# =========================
# 로그 출력
# =========================
print("\n=========================")
print(f"Saved mean RAW to: {save_path}")
print(f"dtype: {mean_img_u16.dtype}")
print(f"shape: {mean_img_u16.shape}")
print(f"min: {mean_img_u16.min()}")
print(f"max: {mean_img_u16.max()}")
print(f"mean: {mean_img.mean():.4f}")
print(f"std : {mean_img.std():.4f}")

Found directories: 30
[Warning] /database2/iyj0121/samsung/251205/scene_000001/SN_20251205_140318 x=09 -> found 0 files
[Warning] /database2/iyj0121/samsung/251205/scene_000001/SN_20251205_140318 x=10 -> found 0 files
[Warning] /database2/iyj0121/samsung/251205/scene_000001/SN_20251205_140323 x=09 -> found 0 files
[Warning] /database2/iyj0121/samsung/251205/scene_000001/SN_20251205_140323 x=10 -> found 0 files
[Warning] /database2/iyj0121/samsung/251205/scene_000001/SN_20251205_140328 x=09 -> found 0 files
[Warning] /database2/iyj0121/samsung/251205/scene_000001/SN_20251205_140328 x=10 -> found 0 files
[Warning] /database2/iyj0121/samsung/251205/scene_000001/SN_20251205_140335 x=09 -> found 0 files
[Warning] /database2/iyj0121/samsung/251205/scene_000001/SN_20251205_140335 x=10 -> found 0 files
[Warning] /database2/iyj0121/samsung/251205/scene_000001/SN_20251205_140343 x=09 -> found 0 files
[Warning] /database2/iyj0121/samsung/251205/scene_000001/SN_20251205_140343 x=10 -> found 0 file